[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/15_evaluacion_retrieval/15_evaluacion_retrieval.ipynb)

# Módulo 15 · Evaluación de embeddings y retrieval
**Tokenización y Embeddings: de BPE a los modelos modernos**

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) |
| **GPU recomendada** | No necesaria (CPU; ≈ 0 unidades de Colab). Una celda opcional usa un embedder real si hay red |
| **Prerrequisitos** | 00 (BM25 + mini golden set), 10 (similitud y pooling), 12 (embedders y MTEB), 13 (híbrido, RRF), 14 (recall en ANN) |
| **Hilo conductor** | Políglota Labs necesita saber **si el nuevo embedder es mejor de verdad** antes de migrar el índice |

> 🧭 **Ruta núcleo vs ampliación.** Núcleo (≈ 3 h, lo que usa el proyecto): secciones 1–6 (métricas, benchmarks, bootstrap y tests, golden sets) y 11 (análisis de errores: TODO 5 del proyecto).
> Ampliación (≈ 2 h): secciones 7–10 (LLM-as-judge, límite teórico de la dimensión, contaminación, drift).
> Si vas justo de tiempo, haz el núcleo y salta a la autoevaluación.

## 🎯 Objetivos de aprendizaje
Al terminar podrás:
1. **Implementar** nDCG@k, Recall@k y MRR@k (con relevancias graduadas, empates, duplicados y queries sin relevantes) y **verificarlas** con tests y contra `ranx`.
2. **Calcular a mano** un nDCG y **explicar** por qué cada métrica premia cosas distintas (posición vs cobertura vs primer acierto).
3. **Construir** un golden set propio (muestreo estratificado, protocolo de anotación, acuerdo entre anotadores κ) y **estimar** cuántas queries necesitas.
4. **Cuantificar la incertidumbre** con bootstrap (IC al 95 %), comparar sistemas con tests pareados y corregir comparaciones múltiples (Holm).
5. **Diseñar** un protocolo de *LLM-as-judge* para relevancia y **detectar sus sesgos** (inflado por solapamiento léxico, sesgo de posición/longitud).
6. **Explicar y reproducir** el límite teórico de los embeddings de dimensión fija (Weller et al., LIMIT) con un experimento de embeddings libres.
7. **Detectar** contaminación y sobreajuste al leaderboard, y **monitorizar** drift con un set canario.
8. **Hacer análisis de errores** por *slices* y convertir los fallos en acciones (tokenización, híbrido, datos).

## 🔁 Conexión con módulos anteriores
Antes de empezar, intenta responder sin mirar (recuperación activa):

<details><summary><b>1.</b> (Módulo 00) En el baseline BM25 de Políglota, ¿por qué reportamos nDCG@10 y no solo «accuracy»?</summary>

Porque la tarea es *ranking*: importa **en qué posición** aparece lo relevante, no solo si aparece. La accuracy ni siquiera está definida para una lista ordenada con varios relevantes.
</details>

<details><summary><b>2.</b> (Módulo 10) Con vectores L2-normalizados, ¿qué relación hay entre coseno, producto escalar y distancia L2? ¿Por qué importa al evaluar?</summary>

Coseno = producto escalar; y $\lVert a-b\rVert^2 = 2-2\cos(a,b)$. Los tres **ordenan igual**, así que dan el mismo ranking y las mismas métricas. Si olvidas normalizar, el producto escalar favorece a los vectores de mayor norma y el ranking cambia: una causa clásica de «el embedder nuevo es peor» que es solo un bug de evaluación.
</details>

<details><summary><b>3.</b> (Módulo 13) ¿Qué hace Reciprocal Rank Fusion y por qué no necesita calibrar escalas?</summary>

Suma $1/(k+\text{rango})$ sobre las listas de cada retriever. Solo usa **rangos**, por lo que ignora que BM25 devuelve scores en $[0,\infty)$ y el coseno en $[-1,1]$.
</details>

<details><summary><b>4.</b> (Módulo 14) El «recall@10» de un índice ANN (HNSW vs fuerza bruta) y el «Recall@10» de relevancia de este módulo, ¿son la misma cosa?</summary>

**No.** El recall del ANN mide fidelidad al vecino exacto *del mismo embedder*; el de este módulo mide cuántos documentos **relevantes según humanos** recuperas. Un ANN con recall 1.0 sobre un embedder malo sigue siendo malo. Son dos capas de error que se componen.
</details>

<details><summary><b>5.</b> (Módulos 04 y 01, Diagnóstico) Un embedder nuevo sube el nDCG@10 global, pero las queries en español empeoran. Antes de culpar al modelo, ¿qué dos causas del bloque A comprobarías?</summary>

(a) **Truncado por prima de tokens** (04): si el nuevo modelo tiene peor fertilidad en español, los documentos ES se cortan antes en `max_seq_length` y pierden contenido; (b) **normalización asimétrica** (01): texto ES con acentos en NFD en el índice y NFC en las queries (o al revés). Ambas se ven solo **por slice** de idioma, que es lo que construye la sección 11; el número global las esconde.
</details>

## 1. Intuición: evaluar es la parte difícil

Entrenar un embedder es *relativamente* fácil; saber si es **mejor** que el anterior es lo difícil. Tres cosas hacen que una mejora aparente sea falsa:

1. **La métrica mide otra cosa** que tu producto (p. ej. optimizas Recall@100 y al usuario solo le importa el top-3).
2. **El set de queries es pequeño o sesgado**, y la diferencia es ruido (¡o los qrels son incompletos y castigan al sistema nuevo!).
3. **El benchmark está contaminado o sobreajustado**: el modelo vio las preguntas durante el entrenamiento.

**Analogía con recsys.** Si hiciste el curso `recsys-course`, esto es el mismo problema: allí construiste `cinematch_eval.py` y viste que
las *sampled metrics* pueden **invertir** el ranking de modelos y que sin intervalos de confianza no sabes si 0.01 de nDCG es algo. Aquí heredamos esas lecciones
(ver [`recsys-course/02_evaluation`](../../recsys-course/02_evaluation/README.md)) y las adaptamos al retrieval de texto: **no repetimos** la derivación de nDCG/AP/AUC
ni el catálogo beyond-accuracy; nos centramos en lo específico de embeddings (qrels dispersos, benchmarks de texto, LLM-as-judge, límites de dimensión, contaminación).

**Analogía con compresión/ML clásico.** El golden set es tu *test set*; reutilizarlo para elegir modelos lo convierte en *validation set* sin que te des cuenta (el equivalente a hacer *p-hacking*). Un solo número sin barra de error es como reportar un bitrate sin decir cuántos archivos mediste.

> ### 🎨 GEMINI-IMAGEN · ID: `M15-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Pipeline de evaluación de retrieval y dónde se rompe**
>
> **Qué muestra:** Vista general del módulo: golden set (queries + qrels) → run del sistema → métricas por query → agregación con IC y tests; con los 5 puntos de fallo marcados.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Diagrama de cuatro bloques: golden set, sistema, métrica por query y agregación con IC; cinco advertencias numeradas de dónde falla la evaluación.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> 
> Composición de izquierda a derecha en 4 bloques conectados por flechas grises gruesas:
> 1) Bloque azul «GOLDEN SET» con dos cajas internas: «Queries (qid, texto, idioma)» y «Qrels: qid → {doc_id: relevancia}».
> 2) Bloque verde «SISTEMA A EVALUAR» con una caja «Embedder + índice (BM25 / denso / híbrido)» que produce «RUN: qid → lista ordenada de doc_id».
> 3) Bloque azul «MÉTRICA POR QUERY» con tres cajas pequeñas apiladas: «nDCG@10», «Recall@k», «MRR@10» y debajo un vector de números «[0.82, 0.00, 1.00, 0.45, …]».
> 4) Bloque naranja «AGREGACIÓN CON INCERTIDUMBRE» con: «Media + IC 95 % (bootstrap sobre queries)» y «Test pareado A vs B + corrección de Holm».
> Debajo de cada bloque, un pequeño círculo naranja numerado con una advertencia en texto corto:
> ① bajo el bloque 1: «Qrels incompletos (pooling sesgado)».
> ② bajo el bloque 1: «Contaminación: el modelo vio el test».
> ③ bajo el bloque 2: «Empates y duplicados en el ranking».
> ④ bajo el bloque 3: «Queries sin relevantes».
> ⑤ bajo el bloque 4: «Diferencia dentro del ruido».
> Título superior: «Cómo se evalúa un buscador semántico». Texto inferior pequeño en gris: «Cada flecha es una fuente de error».
> Evita: texto ilegible o inventado, ilustraciones decorativas, sombras 3D, más de 4 bloques principales, flechas ambiguas que se crucen.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M15-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Diagrama de cuatro bloques: golden set, sistema, métrica por query y agregación con IC; cinco advertencias numeradas de dónde falla la evaluación.](assets/M15-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> [GOLDEN SET]──►[SISTEMA]──►[MÉTRICA/QUERY]──►[AGREGACIÓN + IC + TESTS]
>  ① qrels incompletos   ③ empates/duplicados  ④ queries sin relevantes  ⑤ diferencia ≈ ruido
>  ② contaminación
> ```

> 👀 **Qué debes observar.** La evaluación no es «un número»: es una *cadena* con cinco puntos donde puedes equivocarte sin que salte ningún error.
> A lo largo del módulo cada sección ataca uno: ①② en las secciones 6 y 10, ③④ en la 3, ⑤ en la 5.

## 2. Entorno, corpus Políglota y sistemas de referencia

In [ ]:
!pip install -q numpy scipy scikit-learn matplotlib pandas torch ranx datasets sentence-transformers

In [ ]:
import os, re, math, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
FAST_DEV_RUN = True          # True: tamaños diminutos (CPU, minutos). False: corpus más grande
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if FAST_DEV_RUN: torch.set_num_threads(1)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch, device = None, "cpu"

AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "axes.titlesize": 11, "axes.labelsize": 10})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · semilla={SEED}")

### Utilidades del corpus Políglota (`carga_corpus`)
Contrato común del curso: `carga_corpus(fast_dev) -> {"docs", "queries", "qrels", "source"}`. Este notebook es autocontenido, así que incluye su copia
(cadena **oficial → espejo → sintético**). ⚠️ *Diferencia respecto al módulo 00:* aquí el sintético tiene **huecos léxicos a propósito** (las queries usan sinónimos raros), porque
un benchmark en el que BM25 saca 0.98 no sirve para comparar sistemas.

In [ ]:
# Corpus SINTÉTICO de Políglota (último recurso, mismo esquema que el real): 12 temas x 3 facetas.
# Por faceta: términos ES (5), EN (5), identificadores de código (3). Los 3 primeros términos son "centrales"
# (salen mucho en los documentos); los 2 últimos son sinónimos raros: las queries los usan -> hueco léxico.
TEMAS = {
 "cache": (["caché", "memoria", "datos"], ["cache", "memory", "data"], [
   ("ttl", "expiración|ttl|vencimiento|caducidad|refrescar", "expiration|ttl|timeout|lifetime|refresh", "set_ttl|expire_at|touch_key"),
   ("invalidacion", "invalidación|clave|purgar|obsoleto|desalojar", "invalidation|key|purge|stale|flush", "invalidate|purge_keys|delete_pattern"),
   ("evicion", "política|lru|desalojo|capacidad|recorte", "policy|lru|eviction|capacity|trim", "lru_evict|max_entries|pop_oldest")]),
 "basedatos": (["base", "datos", "consulta"], ["database", "data", "query"], [
   ("indices", "índice|b-tree|escaneo|selectividad|acelerar", "index|btree|scan|selectivity|speedup", "create_index|explain_plan|seq_scan"),
   ("transacciones", "transacción|commit|aislamiento|rollback|atomicidad", "transaction|commit|isolation|rollback|atomicity", "begin_tx|commit_tx|rollback_tx"),
   ("migraciones", "migración|esquema|columna|versionar|alterar", "migration|schema|column|versioning|alter", "run_migration|add_column|schema_version")]),
 "auth": (["usuario", "acceso", "seguridad"], ["user", "access", "security"], [
   ("tokens", "token|jwt|firma|sesión|testigo", "token|jwt|signature|session|bearer", "verify_jwt|sign_token|refresh_token"),
   ("oauth", "oauth|permiso|consentimiento|proveedor|delegar", "oauth|scope|consent|provider|delegate", "oauth_callback|request_scope|exchange_code"),
   ("hash", "contraseña|hash|sal|bcrypt|credencial", "password|hash|salt|bcrypt|credential", "hash_password|check_password|gen_salt")]),
 "contenedores": (["contenedor", "servicio", "despliegue"], ["container", "service", "deployment"], [
   ("imagenes", "imagen|capa|dockerfile|construir|empaquetar", "image|layer|dockerfile|build|package", "docker_build|pull_image|tag_image"),
   ("redes", "red|puerto|puente|exponer|conectar", "network|port|bridge|expose|connect", "create_network|publish_port|attach_bridge"),
   ("volumenes", "volumen|montaje|persistente|disco|compartir", "volume|mount|persistent|disk|share", "mount_volume|bind_path|prune_volumes")]),
 "testing": (["prueba", "código", "calidad"], ["test", "code", "quality"], [
   ("unitarias", "unitaria|aserción|fixture|función|verificar", "unit|assertion|fixture|function|verify", "assert_equal|run_unit|make_fixture"),
   ("mocks", "simulacro|mock|dependencia|sustituir|doble", "mock|stub|dependency|replace|double", "patch_call|make_mock|reset_mock"),
   ("cobertura", "cobertura|línea|rama|informe|medir", "coverage|line|branch|report|measure", "run_coverage|branch_report|min_cover")]),
 "concurrencia": (["hilo", "tarea", "ejecución"], ["thread", "task", "execution"], [
   ("hilos", "hilo|gil|paralelo|pool|lanzar", "thread|gil|parallel|pool|spawn", "start_thread|join_all|pool_map"),
   ("asincrono", "asíncrono|corrutina|await|bucle|esperar", "async|coroutine|await|loop|wait", "run_loop|gather_tasks|await_all"),
   ("bloqueos", "bloqueo|candado|interbloqueo|exclusión|sincronizar", "lock|mutex|deadlock|exclusion|synchronize", "acquire_lock|release_lock|try_lock")]),
 "api": (["api", "petición", "recurso"], ["api", "request", "resource"], [
   ("paginacion", "paginación|cursor|página|límite|desplazamiento", "pagination|cursor|page|limit|offset", "next_cursor|page_size|fetch_page"),
   ("versionado", "versión|compatibilidad|ruta|migrar|deprecar", "version|compatibility|route|migrate|deprecate", "api_version|route_v2|sunset_header"),
   ("limitacion", "cuota|tasa|límite|429|estrangular", "quota|rate|limit|429|throttle", "rate_limit|retry_after|token_bucket")]),
 "observabilidad": (["sistema", "registro", "evento"], ["system", "log", "event"], [
   ("logs", "log|nivel|formato|rotar|mensaje", "log|level|format|rotate|message", "get_logger|set_level|rotate_logs"),
   ("metricas", "métrica|contador|histograma|panel|alerta", "metric|counter|histogram|dashboard|alert", "inc_counter|observe_hist|export_metrics"),
   ("trazas", "traza|span|latencia|distribuido|correlación", "trace|span|latency|distributed|correlation", "start_span|inject_trace|trace_id")]),
 "git": (["repositorio", "cambio", "rama"], ["repository", "change", "branch"], [
   ("ramas", "rama|checkout|puntero|crear|cambiar", "branch|checkout|pointer|create|switch", "git_branch|git_checkout|list_branches"),
   ("merge", "fusión|merge|conflicto|resolver|integrar", "merge|conflict|resolve|integrate|combine", "git_merge|resolve_conflict|abort_merge"),
   ("rebase", "rebase|historial|reescribir|reaplicar|linealizar", "rebase|history|rewrite|replay|linearize", "git_rebase|squash_commits|pick_commit")]),
 "ml": (["modelo", "entrenamiento", "datos"], ["model", "training", "data"], [
   ("gradiente", "gradiente|descenso|pérdida|paso|optimizar", "gradient|descent|loss|step|optimize", "grad_step|zero_grad|clip_norm"),
   ("sobreajuste", "sobreajuste|regularización|generalizar|memorizar|dropout", "overfitting|regularization|generalize|memorize|dropout", "add_dropout|l2_penalty|early_stop"),
   ("validacion", "validación|pliegue|cruzada|partición|estimar", "validation|fold|cross|split|estimate", "kfold_split|val_score|holdout")]),
 "red": (["red", "paquete", "conexión"], ["network", "packet", "connection"], [
   ("tcp", "tcp|segmento|retransmisión|ventana|fiable", "tcp|segment|retransmission|window|reliable", "tcp_connect|set_window|resend_segment"),
   ("dns", "dns|dominio|resolución|registro|nombre", "dns|domain|resolution|record|name", "resolve_host|lookup_a|flush_dns"),
   ("tls", "tls|certificado|cifrado|apretón|confianza", "tls|certificate|encryption|handshake|trust", "tls_handshake|load_cert|verify_chain")]),
 "memoria": (["memoria", "proceso", "objeto"], ["memory", "process", "object"], [
   ("gc", "recolector|basura|referencia|ciclo|liberar", "collector|garbage|reference|cycle|free", "gc_collect|weak_ref|ref_count"),
   ("virtual", "virtual|página|dirección|paginación|fallo", "virtual|page|address|paging|fault", "map_page|page_fault|swap_out"),
   ("fugas", "fuga|crecimiento|perfilar|heap|retención", "leak|growth|profile|heap|retention", "heap_snapshot|track_alloc|find_leak")]),
}

In [ ]:
import os, random, re

PLANT_ES = ["Esta guía explica cómo {a} influye en {b} cuando se trabaja con {c}.",
            "Para resolver el problema de {a}, revisa primero {b} y luego {c}.",
            "En producción conviene vigilar {a}, porque {b} cambia el comportamiento de {c}.",
            "Un error habitual es mezclar {a} con {b} sin entender {c}.",
            "La documentación recomienda medir {a} antes de tocar {b} o {c}.",
            "Ejemplo práctico: ajustar {a} reduce el coste de {b} en {c}."]
PLANT_EN = ["This guide explains how {a} affects {b} when working with {c}.",
            "To solve the problem with {a}, first check {b} and then {c}.",
            "In production you should watch {a}, because {b} changes the behavior of {c}.",
            "A common mistake is mixing {a} with {b} without understanding {c}.",
            "The documentation recommends measuring {a} before touching {b} or {c}.",
            "Practical example: tuning {a} lowers the cost of {b} in {c}."]
QS_ES = ["cómo gestionar {s}", "problemas con {s}", "guía de {s}", "qué hacer con {s}"]
QS_EN = ["how to handle {s}", "problems with {s}", "guide to {s}", "what to do about {s}"]

def _slot(rng, core, rare, shared, other):
    r = rng.random()
    return rng.choice(core) if r < .62 else rng.choice(shared) if r < .80 else rng.choice(rare) if r < .86 else rng.choice(other)

def genera_sintetico(n_temas: int = 8, docs_por_faceta: int = 4, seed: int = 42) -> dict:
    """Mismo esquema que el corpus real. Relevancia: docs de la misma faceta e idioma (code: faceta, queries 'code')."""
    rng = random.Random(seed)
    nombres = list(TEMAS)[:n_temas]
    docs, queries, qrels = [], [], {}
    for t in nombres:
        es_sh, en_sh, facetas = TEMAS[t]
        for fid, es, en, code in facetas:
            otros = [x for n in nombres if n != t for x in TEMAS[n][2][0][1].split("|")[:3]]
            otros_en = [x for n in nombres if n != t for x in TEMAS[n][2][0][2].split("|")[:3]]
            es, en, code = es.split("|"), en.split("|"), code.split("|")
            rel = {"es": {}, "en": {}, "code": {}}
            for lang, terms, shared, oth, plant in (("es", es, es_sh, otros, PLANT_ES), ("en", en, en_sh, otros_en, PLANT_EN)):
                for i in range(docs_por_faceta):
                    sents = [rng.choice(plant).format(a=_slot(rng, terms[:3], terms[3:], shared, oth),
                                                      b=_slot(rng, terms[:3], terms[3:], shared, oth),
                                                      c=_slot(rng, terms[:3], terms[3:], shared, oth)) for _ in range(3)]
                    did = f"{lang}-{t}.{fid}-{i}"
                    docs.append({"id": did, "lang": lang, "text": " ".join(sents)}); rel[lang][did] = 1
            for i in range(max(2, docs_por_faceta - 1)):
                fn = rng.choice(code); aux = rng.choice(code); w = rng.choice(en[:3])
                did = f"code-{t}.{fid}-{i}"
                docs.append({"id": did, "lang": "code", "text": f"def {fn}(x, n={rng.randint(2, 9)}):\n    \"\"\"{w} helper for {rng.choice(en_sh)}\"\"\"\n    return {aux}(x, {rng.choice(code)})"})
                rel["code"][did] = 1
            for lang, terms, qs in (("es", es, QS_ES), ("en", en, QS_EN), ("code", en, QS_EN)):
                s = " ".join(([rng.choice(terms[:3])] if rng.random() < .5 else []) + rng.sample(terms[3:], 2))   # 50%: 1 central + 2 raros
                qid = f"q-{lang}-{t}.{fid}"
                queries.append({"qid": qid, "lang": lang, "text": rng.choice(qs).format(s=s) + (" function" if lang == "code" else "")})
                qrels[qid] = dict(rel[lang])
    rng.shuffle(docs)
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "synthetic"}

In [ ]:
def _carga_real(n_docs: int, seed: int = 42):
    """Cadena OFICIAL: HF `wikimedia/wikipedia` (ES/EN, streaming) + CodeSearchNet Python. Lanza excepción si no hay red/HF.
    Espejo: define HF_ENDPOINT=https://hf-mirror.com ANTES de importar `datasets` (reinicia el runtime) y repite."""
    from datasets import load_dataset
    docs, queries, qrels = [], [], {}
    for lang, cfg in (("es", "20231101.es"), ("en", "20231101.en")):
        ds = load_dataset("wikimedia/wikipedia", cfg, split="train", streaming=True).shuffle(seed=seed, buffer_size=2000)
        got = 0
        for row in ds:
            body = re.sub(r"\s+", " ", row["text"].split("\n\n")[0]).strip()
            if len(body) < 200:
                continue
            did, qid = f"{lang}-{row['id']}", f"q-{lang}-{row['id']}"
            docs.append({"id": did, "lang": lang, "text": body[:700]})
            queries.append({"qid": qid, "lang": lang, "text": row["title"]}); qrels[qid] = {did: 1}
            got += 1
            if got >= n_docs: break
    cs = load_dataset("code-search-net/code_search_net", "python", split="test", streaming=True)
    got = 0
    for i, row in enumerate(cs):
        doc_str = (row.get("func_documentation_string") or "").strip().split("\n")[0]
        code = row.get("func_code_string") or row.get("whole_func_string") or ""
        if len(doc_str) < 25 or not code: continue
        did, qid = f"code-{i}", f"q-code-{i}"
        docs.append({"id": did, "lang": "code", "text": code[:700]})
        queries.append({"qid": qid, "lang": "code", "text": doc_str}); qrels[qid] = {did: 1}
        got += 1
        if got >= n_docs: break
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "real"}

def carga_corpus(fast_dev: bool = True) -> dict:
    """Contrato Políglota: {"docs":[{id,lang,text}], "queries":[{qid,lang,text}], "qrels":{qid:{doc_id:1}}, "source":"real"|"synthetic"}.
    Cadena: oficial (HF) -> espejo (HF_ENDPOINT) -> sintético. NOTA: la versión real usa queries = título del artículo /
    docstring (golden set auto-supervisado, más fácil que uno humano); el sintético usa facetas con huecos léxicos."""
    if os.environ.get("POLIGLOTA_FORCE_SYNTHETIC") != "1":
        try:
            return _carga_real(n_docs=150 if fast_dev else 1500)
        except Exception as e:  # sin red, proxy, dataset movido...
            print(f"⚠️ Datos reales no disponibles ({type(e).__name__}: {str(e)[:80]}) -> corpus SINTÉTICO.")
    return genera_sintetico(n_temas=8 if fast_dev else 12)

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids, doc_texts = [d["id"] for d in docs], [d["text"] for d in docs]
q_lang = {q["qid"]: q["lang"] for q in queries}
print("fuente:", corpus["source"], "| docs:", len(docs), "| queries:", len(queries))
print(pd.DataFrame(docs).lang.value_counts().to_dict(), "docs por idioma;", pd.Series(q_lang).value_counts().to_dict(), "queries por idioma")
print("relevantes por query (media):", np.mean([len(v) for v in qrels.values()]).round(2))
for q in queries[:3]: print(" ", q["qid"], "→", q["text"])

## 3. Métricas: qué miden y cómo implementarlas sin errores

### 3.1 Definiciones (con cada símbolo)
Para una query $q$, un ranking $\pi = (d_1, d_2, \dots)$ y relevancias $\text{rel}(d)\ge 0$ (0 = no relevante o no juzgado):

$$\text{DCG@}k=\sum_{i=1}^{k}\frac{g(\text{rel}(d_i))}{\log_2(i+1)},\qquad \text{nDCG@}k=\frac{\text{DCG@}k}{\text{IDCG@}k}$$

- $g(r)=r$ (ganancia lineal, **la de `trec_eval`/BEIR/MTEB**) o $g(r)=2^r-1$ (Burges; premia mucho más lo muy relevante).
- $\text{IDCG@}k$ es el DCG del **mejor ranking posible** con los documentos juzgados (se ordenan las relevancias de mayor a menor y se toman $k$).
- El descuento $1/\log_2(i+1)$ vale $1$ en la posición 1, $0.63$ en la 2, $0.5$ en la 3… y $0.29$ en la 10.

$$\text{Recall@}k=\frac{|\{d\in\pi_{1..k}:\text{rel}(d)>0\}|}{|\{d:\text{rel}(d)>0\}|},\qquad \text{MRR@}k=\frac{1}{\text{rango del 1.º relevante}}\ (\text{0 si no está en el top-}k)$$

- **nDCG** premia *posición* y admite relevancia graduada → métrica principal de BEIR/MTEB-retrieval.
- **Recall@k** premia *cobertura*: crítico si hay un reranker después (pipeline de dos etapas: Recall@100 del primero acota lo que el segundo puede lograr).
- **MRR** solo mira el *primer acierto*: útil para QA con una respuesta, ciego a todo lo demás.

### 3.2 Ejemplo resuelto a mano
Ranking `[d3, d1, d9, d2, d7]`, relevancias `{d1: 3, d2: 2, d5: 1}` ($d_5$ no aparece en el ranking). Con $k=5$:

- $\text{DCG}=\frac{3}{\log_2 3}+\frac{2}{\log_2 5}$ (d1 en posición 2, d2 en la 4) $=1.893+0.861=2.754$
- $\text{IDCG}=\frac{3}{\log_2 2}+\frac{2}{\log_2 3}+\frac{1}{\log_2 4}=3+1.262+0.5=4.762$
- $\text{nDCG@5}=0.578$ · $\text{Recall@5}=2/3$ · $\text{MRR}=1/2$

### 3.3 Implementación (versión canónica del curso)
Las convenciones son las de `trec_eval`/`ranx`; las decisiones sutiles están en los docstrings y las verificamos con tests justo después.

In [ ]:
import math
import numpy as np

def _dedup(ranked_ids):
    """Conserva la primera aparición: un doc repetido NO puntúa dos veces."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d); out.append(d)
    return out

def _gain(r: float, exp_gain: bool) -> float:
    return (2.0 ** r - 1.0) if exp_gain else float(r)

def dcg_at_k(ranked_ids, rels: dict, k: int = 10, exp_gain: bool = False) -> float:
    """DCG@k = sum_i gain(rel_i) / log2(i+1), i = 1..k. rels: doc_id -> relevancia (>0 relevante; ausente = 0)."""
    return sum(_gain(rels.get(d, 0), exp_gain) / math.log2(i + 2)
               for i, d in enumerate(_dedup(ranked_ids)[:k]) if rels.get(d, 0) > 0)

def ndcg_at_k(ranked_ids, rels: dict, k: int = 10, exp_gain: bool = False) -> float:
    """nDCG@k = DCG@k / IDCG@k. IDCG usa los mejores k de TODOS los juzgados. Sin relevantes -> 0.0 (convención)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(_gain(r, exp_gain) / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k, exp_gain) / idcg

def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """|relevantes en top-k| / |relevantes|. Sin relevantes -> 0.0."""
    relevant = {d for d, r in rels.items() if r > 0}
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant) if relevant else 0.0

def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k; 0 si no hay ninguno."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0

def precision_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Divide SIEMPRE por k (como trec_eval), aunque la lista sea más corta."""
    return sum(rels.get(d, 0) > 0 for d in _dedup(ranked_ids)[:k]) / k

def rank_from_scores(scores, doc_ids, k=None):
    """Orden por score desc; empates -> orden del corpus (determinista). Sin esto, 'argsort' decide por ti."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    return [doc_ids[i] for i in (order[:k] if k else order)]

METRICS = {"ndcg": ndcg_at_k, "recall": recall_at_k, "mrr": mrr_at_k, "precision": precision_at_k}

def evaluate_per_query(run: dict, qrels: dict, k: int = 10, metrics=("ndcg", "recall", "mrr")) -> dict:
    """run: {qid: [doc_id ordenados]}. Devuelve {"qids": [...], metric: np.ndarray}. Excluye queries sin relevantes
    (convención trec_eval); una query sin ranking en `run` puntúa 0 (NO se descarta: sería hacer trampa)."""
    qids = [q for q, rels in qrels.items() if any(r > 0 for r in rels.values())]
    out = {"qids": qids}
    for m in metrics:
        out[m] = np.array([METRICS[m](run.get(q, []), qrels[q], k) for q in qids])
    return out

def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    per = evaluate_per_query(run, qrels, k)
    return {f"{m}@{k}": float(per[m].mean()) for m in ("ndcg", "recall", "mrr")} | {"n_queries": len(per["qids"])}

In [ ]:
ranked = ["d3", "d1", "d9", "d2", "d7"]; rels = {"d1": 3, "d2": 2, "d5": 1}
dcg_mano  = 3 / math.log2(3) + 2 / math.log2(5)
idcg_mano = 3 / math.log2(2) + 2 / math.log2(3) + 1 / math.log2(4)
print(f"a mano:  DCG={dcg_mano:.3f}  IDCG={idcg_mano:.3f}  nDCG@5={dcg_mano/idcg_mano:.3f}")
print(f"código:  nDCG@5={ndcg_at_k(ranked, rels, 5):.3f}  Recall@5={recall_at_k(ranked, rels, 5):.3f}  MRR@5={mrr_at_k(ranked, rels, 5):.3f}")

### 3.4 Tests: los bordes donde fallan las implementaciones caseras
Cada línea es un bug real que he visto en repos de evaluación: duplicados que puntúan dos veces, `argsort` inestable con empates, divisiones por cero cuando una query no tiene relevantes, `k` mayor que la lista…

In [ ]:
def test_metricas():
    close = lambda a, b: abs(a - b) < 1e-9
    # 1) cálculo a mano
    assert close(ndcg_at_k(ranked, rels, 5), dcg_mano / idcg_mano)
    # 2) ranking ideal => nDCG = 1; ranking peor => estrictamente menor
    ideal = ["d1", "d2", "d5", "x"]; assert close(ndcg_at_k(ideal, rels, 10), 1.0)
    assert ndcg_at_k(["x", "d1", "d2", "d5"], rels, 10) < 1.0
    # 3) queries sin relevantes: 0.0 y sin excepción (el agregador decide si las excluye)
    assert ndcg_at_k(ranked, {}, 10) == recall_at_k(ranked, {}, 10) == mrr_at_k(ranked, {}, 10) == 0.0
    # 4) duplicados: un doc repetido NO puntúa dos veces
    assert close(ndcg_at_k(["d1", "d1", "d1"], {"d1": 1, "d2": 1}, 10), ndcg_at_k(["d1"], {"d1": 1, "d2": 1}, 10))
    assert recall_at_k(["d1", "d1"], {"d1": 1, "d2": 1}, 10) == 0.5
    # 5) k mayor que la lista, y k=1
    assert close(recall_at_k(["d1"], {"d1": 1}, 100), 1.0) and mrr_at_k(["x", "d1"], {"d1": 1}, 1) == 0.0
    # 6) relevancia graduada importa: invertir [3,2] cambia el nDCG; ganancia exponencial amplifica la diferencia
    a, b = ["d1", "d2"], ["d2", "d1"]
    assert ndcg_at_k(a, rels, 2) > ndcg_at_k(b, rels, 2)
    assert (ndcg_at_k(a, rels, 2, exp_gain=True) - ndcg_at_k(b, rels, 2, exp_gain=True)) > (ndcg_at_k(a, rels, 2) - ndcg_at_k(b, rels, 2))
    # 7) Recall@k es monótona en k; precision divide por k siempre
    assert all(recall_at_k(ranked, rels, k) <= recall_at_k(ranked, rels, k + 1) for k in range(1, 6))
    assert precision_at_k(["d1"], {"d1": 1}, 10) == 0.1
    # 8) empates: orden determinista = orden del corpus (no depende del algoritmo de argsort)
    ids = ["a", "b", "c", "d"]; sc = np.array([1.0, 2.0, 2.0, 2.0])
    assert rank_from_scores(sc, ids) == ["b", "c", "d", "a"] == rank_from_scores(sc.copy(), ids)
    # 9) un doc relevante con relevancia 0 explícita NO cuenta como relevante
    assert recall_at_k(["d1"], {"d1": 0, "d2": 1}, 5) == 0.0
    return "✅ 9 grupos de tests OK"
print(test_metricas())

> ⚠️ **Cuidado con los empates.** Con modelos cuantizados o scores enteros (BM25 con documentos idénticos, embeddings binarios con distancia de Hamming) los empates abundan.
> Si ordenas con `np.argsort` sin `kind="stable"`, **dos implementaciones (o el mismo código con otro orden de entrada o versión de NumPy) pueden dar métricas distintas**. Fijar el desempate (aquí: orden del corpus) es parte del protocolo.

### 3.5 Tres sistemas para evaluar
Necesitamos algo que evaluar. Sin descargas: **BM25** (léxico, implementación propia), **LSI-48** (TF-IDF + SVD: un «embedder» denso de CPU) y su fusión **RRF**.
Si tienes red y `sentence-transformers`, la celda siguiente añade un embedder real (`multilingual-e5-small`) como cuarto sistema.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def tok_simple(text: str) -> list[str]:
    """Token léxico: minúsculas + regex de palabras (snake_case queda como UN solo token; lo veremos fallar en código)."""
    return re.findall(r"\w+", text.lower())

def tok_code(text: str) -> list[str]:
    """Igual, pero parte snake_case y camelCase (set_ttl -> set, ttl). Es tokenización con criterio de dominio."""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text).replace("_", " ")
    return re.findall(r"[^\W_]+", text.lower())

class BM25:
    """BM25 (Robertson/Lucene idf) con matriz dispersa. k1=1.2, b=0.75."""
    def __init__(self, tokenizer=tok_simple, k1=1.2, b=0.75):
        self.tok, self.k1, self.b = tokenizer, k1, b
    def fit(self, texts, ids):
        self.ids = list(ids)
        self.cv = CountVectorizer(tokenizer=self.tok, lowercase=False, token_pattern=None)
        X = self.cv.fit_transform(texts).tocsr().astype(float)
        self.N = X.shape[0]; self.dl = np.asarray(X.sum(1)).ravel(); self.avg = self.dl.mean()
        df = np.asarray((X > 0).sum(0)).ravel()
        self.idf = np.log(1 + (self.N - df + 0.5) / (df + 0.5))
        self.X = X.tocsc()
        return self
    def scores(self, query: str) -> np.ndarray:
        s = np.zeros(self.N)
        for j in set(self.cv.transform([query]).indices):
            col = self.X[:, j].toarray().ravel()
            s += self.idf[j] * col * (self.k1 + 1) / (col + self.k1 * (1 - self.b + self.b * self.dl / self.avg))
        return s

class LSI:
    """Retriever 'denso' sin descargas: TF-IDF sublineal + SVD truncada (LSA) + coseno. Hace de embedder de CPU."""
    def __init__(self, dim=48, seed=42):
        self.dim, self.seed = dim, seed
    def fit(self, texts, ids):
        self.ids = list(ids)
        self.tf = TfidfVectorizer(sublinear_tf=True)
        T = self.tf.fit_transform(texts)
        self.svd = TruncatedSVD(min(self.dim, T.shape[1] - 1), random_state=self.seed)
        self.Z = self._norm(self.svd.fit_transform(T))
        return self
    @staticmethod
    def _norm(Z): return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)
    def scores(self, query: str) -> np.ndarray:
        return self.Z @ self._norm(self.svd.transform(self.tf.transform([query])))[0]

def rrf_scores(score_lists, k: int = 60) -> np.ndarray:
    """Reciprocal Rank Fusion: sum 1/(k + rango). Ignora la escala de cada score (ventaja y límite)."""
    out = np.zeros(len(score_lists[0]))
    for s in score_lists:
        order = np.lexsort((np.arange(len(s)), -np.asarray(s)))
        out[order] += 1.0 / (k + np.arange(len(s)) + 1)
    return out

def build_run(scorer, queries, doc_ids, topn: int = 100) -> dict:
    """scorer(texto_query) -> vector de scores alineado con doc_ids. Devuelve run {qid: [doc_ids top-n]}."""
    return {q["qid"]: rank_from_scores(scorer(q["text"]), doc_ids, topn) for q in queries}

In [ ]:
bm25 = BM25(tok_simple).fit(doc_texts, doc_ids)
lsi = LSI(dim=48, seed=SEED).fit(doc_texts, doc_ids)
SISTEMAS = {
    "BM25": bm25.scores,
    "LSI-48": lsi.scores,
    "RRF(BM25+LSI)": lambda q: rrf_scores([bm25.scores(q), lsi.scores(q)]),
}

def try_sentence_transformer(name="intfloat/multilingual-e5-small"):
    """Opcional: embedder real. Devuelve scorer o None si no hay red/librería (nunca rompe el notebook)."""
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(name, device=device)
        E = m.encode(["passage: " + t for t in doc_texts], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        return lambda q: E @ m.encode(["query: " + q], normalize_embeddings=True)[0]
    except Exception as e:
        print(f"ℹ️ Embedder real no disponible ({type(e).__name__}); seguimos con BM25/LSI/RRF.")
        return None

if not FAST_DEV_RUN or os.environ.get("TRY_ST") == "1":
    st = try_sentence_transformer()
    if st: SISTEMAS["e5-small"] = st

RUNS = {n: build_run(f, queries, doc_ids, topn=100) for n, f in SISTEMAS.items()}
resumen = pd.DataFrame({n: evaluate(r, qrels, 10) for n, r in RUNS.items()}).T.round(3)
resumen

### 3.6 Verificación contra `ranx`
Nunca confíes en tu propia implementación de una métrica sin contrastarla con una referencia independiente. `ranx` (Bassani, ECIR 2022) es una librería ampliamente usada; debe coincidir al 1e-9.

In [ ]:
try:
    from ranx import Qrels, Run, evaluate as ranx_evaluate
    q_ranx = Qrels({q: {d: int(r) for d, r in rels.items()} for q, rels in qrels.items()})
    for nombre, run in RUNS.items():
        # ranx necesita scores: damos scores decrecientes según la posición del ranking
        r_ranx = Run({q: {d: float(len(lst) - i) for i, d in enumerate(lst)} for q, lst in run.items()})
        ref = ranx_evaluate(q_ranx, r_ranx, ["ndcg@10", "recall@10", "mrr@10"])
        mio = evaluate(run, qrels, 10)
        dif = max(abs(ref["ndcg@10"] - mio["ndcg@10"]), abs(ref["recall@10"] - mio["recall@10"]), abs(ref["mrr@10"] - mio["mrr@10"]))
        print(f"{nombre:16s} máx|ranx - mío| = {dif:.2e}  {'✅' if dif < 1e-9 else '❌'}")
except ImportError:
    print("ranx no instalado: `pip install ranx` para esta verificación (opcional).")

### 3.7 Gráfico: cómo se ven las tres métricas sobre el mismo run

In [ ]:
ks = [1, 2, 3, 5, 10, 20, 50, 100]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
colores = [AZUL, NARANJA, VERDE, GRIS]
for (nombre, run), c in zip(RUNS.items(), colores):
    for a, (m, fn) in zip(ax, (("nDCG@k", ndcg_at_k), ("Recall@k", recall_at_k), ("MRR@k", mrr_at_k))):
        v = [np.mean([fn(run[q], qrels[q], k) for q in qrels if any(qrels[q].values())]) for k in ks]
        a.plot(ks, v, marker="o", ms=3, color=c, label=nombre); a.set_title(m); a.set_xscale("log"); a.set_xlabel("k (escala log)")
ax[0].set_ylabel("valor medio sobre queries"); ax[0].legend(frameon=False, fontsize=8)
fig.suptitle("Las mismas rankings, tres métricas: saturan a ritmos distintos", y=1.03); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) **Recall@k** crece sin parar con $k$ y llega casi a 1 hacia $k=100$: es la métrica del *primer paso* de un pipeline.
> (2) **MRR@k** casi no cambia después de $k\approx 5$: solo mira el primer acierto. (3) **nDCG@k no es monótona** en $k$: puede bajar al principio (al aparecer relevantes de grado bajo detrás de uno muy bien colocado) y sube cuando entran más relevantes. (4) El *orden entre sistemas* puede cambiar según la métrica y el $k$ elegido:
> por eso se fija el protocolo (nDCG@10) **antes** de ver los resultados.

In [ ]:
pos = np.arange(1, 11)
ranking_a = ["x", "r2", "x", "r1", "x", "x", "r3", "x", "x", "x"]   # relevantes (grados 3,2,1) en posiciones 2,4,7
ranking_b = ["r1", "r2", "r3", "x", "x", "x", "x", "x", "x", "x"]   # ranking ideal
rr = {"r1": 3, "r2": 2, "r3": 1}
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(pos, 1 / np.log2(pos + 1), "o-", color=AZUL, label="descuento nDCG  1/log₂(i+1)")
ax[0].plot(pos, 1 / pos, "s--", color=NARANJA, label="descuento MRR/AP  1/i")
ax[0].set_xlabel("posición i"); ax[0].set_ylabel("peso"); ax[0].set_title("El descuento de nDCG es más suave que 1/i"); ax[0].legend(frameon=False)
w = .38
for off, (nom, rk, c) in zip((-w/2, w/2), (("ranking A", ranking_a, ROJO), ("ranking ideal", ranking_b, VERDE))):
    contrib = [rr.get(d, 0) / math.log2(i + 2) for i, d in enumerate(rk)]
    ax[1].bar(pos + off, contrib, w, color=c, label=f"{nom}: nDCG@10={ndcg_at_k(rk, rr, 10):.2f}")
ax[1].set_xlabel("posición"); ax[1].set_ylabel("contribución al DCG"); ax[1].set_title("Dónde se gana el DCG (ganancia 3, 2, 1)"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** A la izquierda, el descuento logarítmico castiga menos las posiciones bajas que $1/i$: nDCG es «más tolerante» que MRR/AP.
> A la derecha, el ranking A pone el documento más relevante (ganancia 3) en la posición 4 en vez de la 1: pierde un tercio del DCG (nDCG 0.67) aunque *recupere los tres relevantes*.
> Recall@10 sería **idéntico (1.0)** para ambos: Recall no ve el orden.

## 4. Benchmarks públicos: qué miden y qué no

| Benchmark | Qué es | Detalles verificados | Cuidado |
|---|---|---|---|
| **BEIR** (Thakur et al., NeurIPS 2021) | 18 datasets heterogéneos de retrieval *zero-shot* (SciFact, FiQA, NFCorpus, TREC-COVID…) | **SciFact**: verificación de afirmaciones científicas, ≈ 5 K abstracts y 300 queries de test; **FiQA**: QA financiero (opiniones), ≈ 57 K documentos y 648 queries de test (BEIR, Tabla 1) | qrels dispersos: muchos relevantes **sin juzgar** (sección 6) |
| **MIRACL** (Zhang et al., 2022) | Retrieval monolingüe en **18 idiomas** (incluye español) con juicios humanos | Wikipedia por idioma; evaluado con nDCG@10 | Solo monolingüe; sin cross-lingual |
| **MTEB** (Muennighoff et al., 2022) | 8 tareas / 58 datasets / 112 idiomas; para retrieval, métrica principal nDCG@10 | Promedia tareas muy distintas | El promedio oculta qué tarea importa a *tu* producto |
| **MMTEB** (Enevoldsen et al., ICLR 2025) | Extensión comunitaria: **> 500 tareas, 250+ idiomas** | Incluye retrieval, instrucciones, código… | Sigue siendo público → contaminable |
| **RTEB** (HF + MTEB, beta oct-2025) | Retrieval con datasets **privados** para medir overfitting | 48 datasets, 28 privados; correlación de rankings con BEIR/MTEB débil (ρ = 0.38) según su paper | 🟡 Joven; los privados no son auditables por ti |

> 📅 *Cifras a oct-2026, tomadas de los papers/blog citados en Referencias; los leaderboards cambian cada semana: no copies números de ahí sin fecha.*

In [ ]:
def carga_beir(nombre: str = "scifact", split: str = "test", max_docs: int = 5000, max_queries: int = 300):
    """BEIR vía Hugging Face (`BeIR/<nombre>` + `BeIR/<nombre>-qrels`). Devuelve el mismo contrato que carga_corpus o None.
    ⚠️ No pude probarlo en mi entorno de desarrollo (HF bloqueado): el esquema es el documentado en las model cards de BeIR."""
    try:
        from datasets import load_dataset
        corp = load_dataset(f"BeIR/{nombre}", "corpus", split="corpus")
        qs = load_dataset(f"BeIR/{nombre}", "queries", split="queries")
        qr = load_dataset(f"BeIR/{nombre}-qrels", split=split)
    except Exception as e:
        print(f"ℹ️ BEIR/{nombre} no disponible ({type(e).__name__}) -> usa el corpus Políglota de arriba.")
        return None
    rel: dict = {}
    for r in qr:
        if r["score"] > 0: rel.setdefault(str(r["query-id"]), {})[str(r["corpus-id"])] = int(r["score"])
    keep = list(rel)[:max_queries]
    need = {d for q in keep for d in rel[q]}
    docs_b = [{"id": str(r["_id"]), "lang": "en", "text": (r["title"] + ". " + r["text"]).strip()} for r in corp]
    docs_b = [d for d in docs_b if d["id"] in need] + [d for d in docs_b if d["id"] not in need][: max(0, max_docs - len(need))]
    qd = {str(r["_id"]): r["text"] for r in qs}
    return {"docs": docs_b, "queries": [{"qid": q, "lang": "en", "text": qd[q]} for q in keep if q in qd],
            "qrels": {q: rel[q] for q in keep}, "source": f"real:BeIR/{nombre}"}

beir = carga_beir("scifact") if not FAST_DEV_RUN or os.environ.get("TRY_BEIR") == "1" else None
if beir:
    b_ids = [d["id"] for d in beir["docs"]]; b_bm25 = BM25(tok_simple).fit([d["text"] for d in beir["docs"]], b_ids)
    b_run = build_run(b_bm25.scores, beir["queries"], b_ids)
    print("BM25 en", beir["source"], evaluate(b_run, beir["qrels"], 10))
else:
    print("BEIR no cargado (FAST_DEV_RUN=True o sin red). Pon FAST_DEV_RUN=False en Colab para probarlo; "
          "referencia pública: BM25 en SciFact ≈ 0.665 nDCG@10 (BEIR, 2021).")

### 4.1 Cómo se corre MTEB «de verdad» (celda opcional)
La librería `mteb` evalúa un modelo en tareas con la métrica oficial. Su API **cambió entre v1 y v2**; verifica con tu versión (`pip show mteb`). La celda está protegida:
no se ejecuta salvo que pongas `RUN_MTEB = True`.

In [ ]:
RUN_MTEB = False   # True en Colab con red: descarga el modelo y los datasets de SciFact/FiQA
if RUN_MTEB:
    import mteb
    modelo = mteb.get_model("intfloat/multilingual-e5-small")
    tareas = mteb.get_tasks(tasks=["SciFact", "FiQA2018"])
    try:
        resultados = mteb.evaluate(modelo, tareas)                        # mteb >= 2.x
    except AttributeError:
        resultados = mteb.MTEB(tasks=tareas).run(modelo, output_folder="mteb_out")   # mteb 1.x
    print(resultados)
else:
    print("RUN_MTEB=False: celda omitida (ejemplo de uso, no se ejecuta).")

## 5. Incertidumbre: bootstrap, tests pareados y comparaciones múltiples

Un nDCG medio de 0.52 vs 0.50 sobre 72 queries, ¿es una mejora? Depende de la **varianza entre queries**. La unidad de muestreo es la *query* (no el par query-documento): las queries son las observaciones independientes.

### 5.1 Bootstrap percentil para la media
1. Tienes $n$ valores por query $x_1..x_n$ (p. ej. nDCG@10 de cada una).
2. Repite $B$ veces: remuestrea $n$ valores **con reemplazo** y calcula su media $\bar x^*_b$.
3. El IC 95 % es el intervalo $[q_{2.5\%},\,q_{97.5\%}]$ de las $\bar x^*_b$.

✍️ **Ejemplo a mano (pareado, 4 queries).** Diferencias B − A por query: $(0.10,\ -0.05,\ 0.20,\ 0.00)$, media $0.0625$. Un remuestreo con reemplazo saca las queries $(1, 1, 3, 4)$ → media $(0.10 + 0.10 + 0.20 + 0)/4 = 0.10$; otro saca $(2, 2, 4, 1)$ → $0.0$. Tras miles de remuestreos, los percentiles 2.5 y 97.5 de esas medias son el IC. Remuestreamos **diferencias por query** (pareado) porque A y B se miden sobre las mismas queries: la dificultad de cada query se cancela, como en el team-draft interleaving del módulo 18 o en el bootstrap del módulo 00. Con solo 4 queries, el IC incluirá el 0 casi seguro: el ejemplo enseña el mecanismo, no una conclusión.

> ### 🎬 GEMINI-VIDEO · ID: `M15-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Bootstrap: remuestrear queries y construir el intervalo de confianza**
>
> **Qué muestra:** Animación del bootstrap percentil: 8 valores de nDCG por query, tres remuestreos con reemplazo, histograma de medias que se va llenando y el IC 95 % sombreado.
> **Formato:** MP4 16:9, 1080p | Duración: 30 s | **Texto alternativo:** Animación en la que ocho valores de nDCG se remuestrean con reemplazo, las medias forman un histograma y se marca el intervalo de confianza al 95 %.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Animación técnica limpia, 16:9, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif, ritmo calmado, sin voz en off, sin música. Texto en pantalla en español.
> Datos exactos: 8 queries con nDCG@10 = [0.00, 0.31, 0.45, 0.50, 0.63, 0.72, 0.89, 1.00]; media = 0.5625.
> Escenas:
> 0–5 s: título «Bootstrap sobre queries». Aparecen 8 barras verticales azules etiquetadas q1…q8 con alturas iguales a los valores anteriores y la línea horizontal gris de la media «media = 0.56».
> 5–12 s: texto «Remuestreo 1 (con reemplazo)». De las 8 barras salen flechas naranjas que "caen" en 8 huecos nuevos; la nueva muestra es «q2, q2, q4, q4, q5, q5, q8, q8»: algunas barras se repiten y otras no aparecen (q1, q3, q6, q7). Se muestra la nueva media «0.61» y un punto naranja cae sobre un eje horizontal inferior (histograma vacío) en 0.61.
> 12–18 s: repetir rápido otros dos remuestreos: «q1, q1, q1, q2, q6, q7, q8, q8» (media 0.49) y «q1, q1, q2, q6, q6, q7, q8, q8» (media 0.58), con puntos que caen sobre el histograma; el contador muestra «B = 3».
> 18–25 s: el contador acelera «B = 100 … 1000 … 2000» y los puntos forman un histograma en forma de campana azul alrededor de 0.56.
> 25–30 s: se sombrean los percentiles 2.5 % y 97.5 % en verde y aparece la etiqueta «IC 95 % ≈ [0.35, 0.77]» sobre el histograma; texto final «la incertidumbre depende del número de queries».
> Evita: voz en off, música, texto ilegible, números distintos a los dados, efectos 3D, transiciones bruscas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M15-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M15-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> q1..q8 = [0.00 0.31 0.45 0.50 0.63 0.72 0.89 1.00]  media 0.56
> remuestreo 1: [q2 q2 q4 q4 q5 q5 q8 q8] -> media 0.61  (punto en el histograma)
> ... B=2000 medias -> campana -> percentiles 2.5% y 97.5% = IC 95% ≈ [0.35, 0.77]
> ```

> 👀 **Qué debes observar.** Algunas queries aparecen repetidas y otras desaparecen en cada remuestreo: eso simula «otro conjunto de queries posible».
> El ancho del histograma de medias es la **incertidumbre por tener solo estas queries**. Con más queries la campana se estrecha como $\sigma/\sqrt n$.

In [ ]:
def bootstrap_ci(values, n_boot: int = 2000, alpha: float = 0.05, seed: int = 42):
    """IC percentil de la media remuestreando QUERIES (la unidad de muestreo es la query, no el par query-doc)."""
    v = np.asarray(values, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    return float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def paired_bootstrap(a, b, n_boot: int = 10000, alpha: float = 0.05, seed: int = 42) -> dict:
    """Test pareado A vs B sobre las MISMAS queries. IC percentil de la diferencia media y p-valor bilateral
    por 'shift' (se centra la diferencia en 0 bajo H0 y se cuenta cuántas veces el remuestreo supera lo observado)."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    null = (d - d.mean())[idx].mean(axis=1)
    p = (np.sum(np.abs(null) >= abs(d.mean())) + 1) / (n_boot + 1)
    return {"diff": float(d.mean()), "ci_low": float(np.quantile(means, alpha / 2)),
            "ci_high": float(np.quantile(means, 1 - alpha / 2)), "p_value": float(p)}

def paired_permutation(a, b, n_perm: int = 10000, seed: int = 42) -> float:
    """Test de aleatorización pareado (sign-flip): bajo H0 el signo de cada diferencia por query es intercambiable."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    signs = np.random.default_rng(seed).choice([-1.0, 1.0], size=(n_perm, len(d)))
    stat = np.abs((signs * d).mean(axis=1))
    return float((np.sum(stat >= abs(d.mean()) - 1e-12) + 1) / (n_perm + 1))

def holm(pvalues: dict, alpha: float = 0.05) -> dict:
    """Corrección de Holm-Bonferroni para comparaciones múltiples. Devuelve {nombre: (p_ajustado, rechaza_H0)}."""
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    m, out, running = len(items), {}, 0.0
    for i, (name, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        out[name] = (running, running < alpha)
    return out

In [ ]:
per = {n: evaluate_per_query(r, qrels, 10) for n, r in RUNS.items()}
qids_eval = per["BM25"]["qids"]
filas = []
for n in RUNS:
    m, lo, hi = bootstrap_ci(per[n]["ndcg"])
    filas.append({"sistema": n, "nDCG@10": m, "IC95_bajo": lo, "IC95_alto": hi, "ancho": hi - lo})
tabla_ic = pd.DataFrame(filas).set_index("sistema").round(3)
display(tabla_ic)

fig, ax = plt.subplots(figsize=(7, 2.8))
for i, (n, r) in enumerate(tabla_ic.iterrows()):
    ax.errorbar(r["nDCG@10"], i, xerr=[[r["nDCG@10"] - r["IC95_bajo"]], [r["IC95_alto"] - r["nDCG@10"]]], fmt="o", color=AZUL, capsize=4)
ax.set_yticks(range(len(tabla_ic))); ax.set_yticklabels(tabla_ic.index); ax.set_xlabel("nDCG@10 (media e IC 95 % bootstrap)")
ax.set_title("Los intervalos se solapan: ¿hay diferencia real?"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Los IC de dos sistemas pueden **solaparse** y aun así uno ser significativamente mejor: ambos sistemas se evalúan sobre *las mismas queries*,
> y las queries difíciles lo son para los dos. El IC de cada media ignora esa correlación; el **test pareado** (siguiente celda) la aprovecha y es mucho más potente.

### 5.2 Test pareado y corrección de Holm
Para comparar A y B sobre las mismas queries se trabaja con las **diferencias por query** $d_i = a_i - b_i$. Dos tests no paramétricos equivalentes:

- **Bootstrap pareado**: IC de $\bar d$ remuestreando queries; p-valor desplazando las diferencias para que $H_0:\bar d=0$ sea cierta.
- **Aleatorización (sign-flip)**: bajo $H_0$ el signo de cada $d_i$ es intercambiable.

Smucker, Allan y Carterette (CIKM 2007) compararon cinco tests sobre runs de TREC y encontraron que bootstrap, randomización y t-test **coinciden casi siempre**; el Wilcoxon y el de signos se desvían. Si comparas $m$ pares de sistemas, el riesgo de falso positivo se acumula: **Holm** controla la tasa de error familiar con más potencia que Bonferroni.

In [ ]:
pares = [("LSI-48", "BM25"), ("RRF(BM25+LSI)", "BM25"), ("RRF(BM25+LSI)", "LSI-48")]
pb, pp, filas = {}, {}, []
for a, b in pares:
    tb = paired_bootstrap(per[a]["ndcg"], per[b]["ndcg"]); pv = paired_permutation(per[a]["ndcg"], per[b]["ndcg"])
    pb[f"{a} vs {b}"] = tb["p_value"]
    filas.append({"comparación": f"{a} − {b}", "Δ nDCG@10": tb["diff"], "IC95_bajo": tb["ci_low"], "IC95_alto": tb["ci_high"], "p (bootstrap)": tb["p_value"], "p (permutación)": pv})
ajustado = holm(pb)
tabla_tests = pd.DataFrame(filas).set_index("comparación")
tabla_tests["p Holm"] = [ajustado[k][0] for k in pb]; tabla_tests["¿significativo (α=.05)?"] = [ajustado[k][1] for k in pb]
tabla_tests.round(4)

> 👀 **Qué debes observar.** La primera fila (LSI−BM25) tiene un p *sin corregir* cercano a 0.03 ("significativo"), pero tras **Holm** (3 comparaciones) el p ajustado sube por encima de 0.05: con 72 queries y tres comparaciones **no puedes afirmarlo**.
> Es lo que pasa en un paper cuando se prueban muchas variantes y solo se reporta la «ganadora». La tercera fila, en cambio, sí resiste la corrección. Con otro corpus los números cambiarán: lo importante es el procedimiento (IC pareado + Holm), no estos valores.

### 5.3 ¿Cuántas queries necesito? (experimento)
El ancho del IC cae como $1/\sqrt n$. Midamos la desviación estándar *real* entre queries de nuestro sistema y deduzcamos el tamaño de golden set necesario para una semi-anchura objetivo.

In [ ]:
x = per["BM25"]["ndcg"]; rng = np.random.default_rng(SEED)
sizes = [n for n in (10, 20, 30, 45, 60, 80, len(x)) if n <= len(x)]
anchos = []
for n in sizes:
    w = [np.subtract(*np.quantile(x[rng.choice(len(x), n, replace=False)][rng.integers(0, n, (400, n))].mean(1), [.975, .025])) for _ in range(20)]
    anchos.append(np.mean(w))
sd = x.std(ddof=1)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(sizes, anchos, "o-", color=AZUL, label="ancho medio del IC95 % (bootstrap)")
nn = np.linspace(min(sizes), max(sizes), 50); ax.plot(nn, 2 * 1.96 * sd / np.sqrt(nn), "--", color=NARANJA, label=r"teoría: $2\cdot1.96\,\sigma/\sqrt{n}$")
ax.set_xlabel("nº de queries en el golden set"); ax.set_ylabel("ancho del IC (nDCG@10)"); ax.legend(frameon=False); ax.set_title("Más queries → intervalos más estrechos"); plt.tight_layout(); plt.show()
print(f"σ entre queries = {sd:.3f}")
for h in (0.05, 0.03, 0.02):
    print(f"  para semi-anchura ±{h:.2f} sobre UNA media: n ≈ {math.ceil((1.96 * sd / h) ** 2)} queries")
d = per["LSI-48"]["ndcg"] - per["BM25"]["ndcg"]
print(f"σ de la DIFERENCIA pareada LSI−BM25 = {d.std(ddof=1):.3f}  (pareada: {'menor' if d.std(ddof=1) < sd else 'similar/mayor'} que σ de una sola métrica)")

> 👀 **Qué debes observar.** El IC medido sigue la curva teórica $\propto 1/\sqrt n$: **para reducir el ancho a la mitad necesitas 4× queries**. Las cifras impresas son *las de tu golden set*, no una regla universal:
> los benchmarks reales (SciFact 300 queries, FiQA 648) tienen σ distinta. Si tu objetivo es detectar mejoras de ±0.01 nDCG, 100 queries no bastan: o amplías el golden set o aceptas decidir solo diferencias grandes.

In [ ]:
# ¿Qué tests mantienen el error tipo I bajo control? Experimento A/A: dos "sistemas" con la MISMA calidad esperada.
rng = np.random.default_rng(SEED); n_q, reps, alpha = 50, 300, .05
from scipy import stats
base = per["BM25"]["ndcg"]; rech = {"bootstrap pareado": 0, "permutación pareada": 0, "t pareado": 0, "t NO pareado (error)": 0}
pot = dict.fromkeys(rech, 0); delta = 0.04
for _ in range(reps):
    x0 = rng.choice(base, n_q)                                  # dificultad de cada query (compartida)
    a = np.clip(x0 + rng.normal(0, .12, n_q), 0, 1); b = np.clip(x0 + rng.normal(0, .12, n_q), 0, 1)
    for tag, (u, v) in (("null", (a, b)), ("alt", (np.clip(a + delta, 0, 1), b))):
        ps = {"bootstrap pareado": paired_bootstrap(u, v, n_boot=400, seed=int(rng.integers(10**9)))["p_value"],
              "permutación pareada": paired_permutation(u, v, n_perm=400, seed=int(rng.integers(10**9))),
              "t pareado": stats.ttest_rel(u, v).pvalue, "t NO pareado (error)": stats.ttest_ind(u, v).pvalue}
        tgt = rech if tag == "null" else pot
        for k, p in ps.items(): tgt[k] += p < alpha
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar(list(rech), [v / reps for v in rech.values()], color=[AZUL, AZUL, AZUL, ROJO]); ax[0].axhline(alpha, color=GRIS, ls="--"); ax[0].set_title("Falsos positivos (A = B): debería ser ≈ 0.05")
ax[1].bar(list(pot), [v / reps for v in pot.values()], color=[VERDE, VERDE, VERDE, ROJO]); ax[1].set_title(f"Potencia con diferencia real Δ={delta}")
for a_ in ax: a_.tick_params(axis="x", rotation=18); a_.set_ylim(0, 1)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Los tres tests pareados mantienen los falsos positivos cerca de 0.05 y tienen **potencia parecida**. Tratar las queries como muestras independientes
> (t no pareado, barra roja) es *conservador* aquí (≈ 0 falsos positivos) pero **pierde casi toda la potencia**: desperdicia la información de que ambos sistemas ven las mismas queries.
> Recomendación práctica: reporta siempre diferencia pareada con IC, no dos IC separados.

## 6. Construir tu propio golden set
Los benchmarks públicos responden «¿es buen embedder en general?». Tu producto necesita un set sobre **tus** documentos y **tus** usuarios. Receta mínima:

1. **Fuente de queries**: logs reales anonimizados > queries escritas por expertos > queries sintéticas generadas por LLM a partir de documentos (útiles para empezar, sesgadas hacia el vocabulario del documento).
2. **Muestreo estratificado** por idioma, longitud y tipo de intención; si no, domina lo frecuente. Reserva una parte (≥ 30 %) como **test ciego** que no se toca al iterar.
3. **Pooling de candidatos**: junta el top-k de **varios sistemas distintos** (léxico + denso + híbrido); anotar solo lo que devuelve un sistema sesga los qrels a su favor (veremos el efecto en 6.2).
4. **Guía de anotación** con grados (0 irrelevante, 1 tangencial, 2 responde parcialmente, 3 responde completamente) y ejemplos límite.
5. **Doble anotación** de una muestra y medir acuerdo con **κ de Cohen**: $\kappa=(p_o-p_e)/(1-p_e)$, donde $p_o$ es el acuerdo observado y $p_e$ el esperado por azar.
6. **Versionar y congelar** (dataset card: fuente, fecha, criterios, κ); recalibrar cuando cambie el corpus o el tráfico.

> ### 🎨 GEMINI-IMAGEN · ID: `M15-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Ciclo de vida de un golden set**
>
> **Qué muestra:** Ciclo en seis pasos: muestrear queries, hacer pooling de candidatos, anotar con guía, medir acuerdo κ, congelar versión y vigilar drift; con el test ciego aparte.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Ciclo de seis pasos para construir y mantener un golden set, con un test ciego en el centro y un bucle de mejora de la guía de anotación.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> Un ciclo circular de seis nodos redondeados conectados por flechas en sentido horario, con estos textos literales:
> 1 (azul) «Muestrear queries (estratificado por idioma e intención)»,
> 2 (azul) «Pooling: top-k de BM25 + denso + híbrido»,
> 3 (naranja) «Anotar con guía (grados 0–3)»,
> 4 (naranja) «Acuerdo entre anotadores: κ de Cohen»,
> 5 (verde) «Congelar versión v1.0 + dataset card»,
> 6 (verde) «Vigilar drift y recalibrar».
> En el centro del ciclo un rectángulo gris con el texto «TEST CIEGO (30 %): no se toca al iterar».
> Entre los nodos 4 y 3 una flecha curva de retorno con la etiqueta «κ < 0.6 → mejorar la guía».
> Título superior: «Ciclo de vida de un golden set».
> Evita: texto ilegible o inventado, ilustraciones de personas, elementos decorativos, más de seis nodos, flechas que se crucen.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M15-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Ciclo de seis pasos para construir y mantener un golden set, con un test ciego en el centro y un bucle de mejora de la guía de anotación.](assets/M15-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> (1) muestrear → (2) pooling → (3) anotar → (4) κ de Cohen → (5) congelar v1.0 → (6) vigilar drift → (1)
>                  [TEST CIEGO 30 %]       κ<0.6 → vuelve a (3)
> ```

> 👀 **Qué debes observar.** El paso 4 es un *bucle de calidad*: si dos humanos no se ponen de acuerdo (κ bajo), ningún embedder podrá «acertar» de forma consistente, y la métrica mide el ruido de la guía de anotación.
> κ > 0.8 se considera acuerdo casi perfecto; entre 0.6 y 0.8, sustancial (escala de Landis y Koch, convención habitual).

### 6.1 Estratificar y repartir dev / test ciego

In [ ]:
def split_estratificado(queries, frac_test=0.3, seed=SEED):
    """Reparto dev/test ciego manteniendo la proporción por idioma (y por faceta temática si el id la lleva)."""
    rng = random.Random(seed); dev, test = [], []
    for lang in sorted({q["lang"] for q in queries}):
        grp = [q for q in queries if q["lang"] == lang]; rng.shuffle(grp)
        cut = int(round(len(grp) * frac_test)); test += grp[:cut]; dev += grp[cut:]
    return dev, test

def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b); labs = np.unique(np.concatenate([a, b]))
    po = np.mean(a == b); pe = sum(np.mean(a == l) * np.mean(b == l) for l in labs)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

dev_q, test_q = split_estratificado(queries)
print(f"dev={len(dev_q)} test_ciego={len(test_q)}", pd.Series([q['lang'] for q in test_q]).value_counts().to_dict())
# Doble anotación simulada: dos anotadores con 90 % de acierto sobre pares (query, doc) reales del pool
rng = np.random.default_rng(SEED); verdad = rng.integers(0, 2, 300)
ann1 = np.where(rng.random(300) < .90, verdad, 1 - verdad); ann2 = np.where(rng.random(300) < .90, verdad, 1 - verdad)
print(f"acuerdo observado={np.mean(ann1==ann2):.2f}  κ={cohen_kappa(ann1, ann2):.2f}  (cada anotador acierta el 90 % frente a la verdad, de forma independiente → acuerdo ≈ 0.82, κ ≈ 0.64)")

### 6.2 El sesgo de pooling: los qrels incompletos castigan a lo nuevo
Imagina que tus qrels se construyeron anotando **solo lo que devolvía BM25** (lo que tenías en 2019). Un embedder denso nuevo recupera documentos genuinamente relevantes que *nadie juzgó*; el evaluador los cuenta como no relevantes. Simulémoslo: conservamos solo los relevantes que BM25 trajo en su top-20.

In [ ]:
def qrels_pooled(qrels, run_ref, depth=20):
    out = {}
    for q, rels in qrels.items():
        top = set(run_ref[q][:depth]); keep = {d: r for d, r in rels.items() if d in top}
        out[q] = keep if keep else {}
    return out

qr_bm25 = qrels_pooled(qrels, RUNS["BM25"], 20)
comp = {}
for n in ("BM25", "LSI-48", "RRF(BM25+LSI)"):
    comp[n] = (evaluate(RUNS[n], qrels, 10)["ndcg@10"], evaluate(RUNS[n], qr_bm25, 10)["ndcg@10"])
perdidos = sum(len(v) for v in qrels.values()) - sum(len(v) for v in qr_bm25.values())
fig, ax = plt.subplots(figsize=(7, 3.4)); x = np.arange(len(comp)); w = .36
ax.bar(x - w/2, [v[0] for v in comp.values()], w, color=VERDE, label="qrels completos (verdad)")
ax.bar(x + w/2, [v[1] for v in comp.values()], w, color=NARANJA, label="qrels agrupados solo desde BM25")
ax.set_xticks(x); ax.set_xticklabels(comp); ax.set_ylabel("nDCG@10"); ax.legend(frameon=False); ax.set_title(f"Sesgo de pooling ({perdidos} relevantes sin juzgar)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con los qrels agrupados desde BM25, **todos los sistemas «suben»** (el IDCG se encoge al desaparecer relevantes) pero BM25 sube **más**: pasa de ser el segundo a parecer el mejor y el orden con LSI **se invierte**. El evaluador premia parecerse al sistema que sirvió para anotar.
> Es exactamente el problema que Arabzadeh et al. (*Shallow pooling for sparse labels*, 2021) documentan en MS MARCO: los rankers neuronales modernos devuelven resultados que los jueces humanos prefieren a los «relevantes» etiquetados.
> El propio paper de BEIR (sección 6) reconoce que varios de sus datasets sufren **sesgo léxico** (se anotaron candidatos de sistemas léxicos) y que en TREC-COVID anotar los juicios que faltaban subió a los modelos densos. **Regla:** al comparar un sistema nuevo, añade *su* top-k a la cola de anotación.

## 7. LLM-as-judge para relevancia (y por qué hay que desconfiar con método)

Anotar a mano 10 000 pares query–documento cuesta semanas. La tentación: pedir a un LLM que etiquete relevancia. La literatura está **dividida** (🟡 debatido):

- **A favor**: Thomas, Spielman, Craswell y Mitra (2023) muestran que prompts afinados contra preferencias de usuarios reales producen etiquetas «tan precisas como las de anotadores humanos» y mucho más baratas.
- **En contra**: Soboroff (2024, *Don't Use LLMs to Make Relevance Judgments*) y Clarke & Dietz (2024) argumentan que, para evaluación tipo TREC, hay riesgos de circularidad (los sistemas que usan LLMs salen favorecidos) y que las métricas basadas en LLM se pueden manipular. Faggioli et al. (2023) proponen un espectro de colaboración humano–máquina.
- **Vulnerabilidad medida**: Alaofi et al. (SIGIR-AP 2024) insertaron las palabras de la query en pasajes irrelevantes y comprobaron que muchos LLMs los etiquetan como relevantes; sus etiquetas «no relevante» son más fiables que las «relevante».
- **Sesgos generales de LLM-judge** (Zheng et al., NeurIPS 2023): de posición, de verbosidad y de auto-preferencia.

**Protocolo razonable hoy:** el LLM como *triaje* (priorizar qué anotar, detectar qrels dudosos), humanos para el set de decisión, y calibración continua: mide κ del LLM vs humanos en una muestra y **actualízala cuando cambies de modelo o de prompt**.

### 7.1 Simulación del mecanismo del sesgo (no es una medición de ningún LLM real)
Modelamos un juez con sensibilidad `tpr` y falsos positivos que **crecen con el solapamiento léxico** con la query (el fallo documentado por Alaofi et al.). Lo aplicamos al *pool* de los tres sistemas y recalculamos las métricas.

In [ ]:
def juez_simulado(q_text, d_text, es_rel, sesgo, rng, tpr=0.9, fpr=0.05):
    """Juez con ruido: P(relevante|rel)=tpr; P(relevante|no rel)=fpr + sesgo * (fracción de términos de la query en el doc)."""
    qt, dt = set(tok_simple(q_text)), set(tok_simple(d_text))
    solap = len(qt & dt) / max(1, len(qt))
    return rng.random() < (tpr if es_rel else min(1.0, fpr + sesgo * solap))

doc_text = dict(zip(doc_ids, doc_texts)); q_text = {q["qid"]: q["text"] for q in queries}
pool = {q: sorted({d for run in RUNS.values() for d in run[q][:10]}) for q in qrels}
def qrels_juez(sesgo, seed=SEED):
    rng = np.random.default_rng(seed); out, tp, fp, fn, tn = {}, 0, 0, 0, 0
    for q, cand in pool.items():
        out[q] = {}
        for d in cand:
            es = qrels[q].get(d, 0) > 0; lab = juez_simulado(q_text[q], doc_text[d], es, sesgo, rng)
            if lab: out[q][d] = 1
            tp += lab and es; fp += lab and not es; fn += (not lab) and es; tn += (not lab) and not es
    n = tp + fp + fn + tn; po = (tp + tn) / n; pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / n**2
    return out, (po - pe) / (1 - pe)
filas = []
for sesgo in (0.0, 0.4, 0.8):
    qj, kappa = qrels_juez(sesgo)
    for n in ("BM25", "LSI-48", "RRF(BM25+LSI)"):
        filas.append({"sesgo": sesgo, "sistema": n, "nDCG@10 (juez)": evaluate(RUNS[n], qj, 10)["ndcg@10"], "κ juez vs verdad": kappa})
jdf = pd.DataFrame(filas); verdad_nd = {n: evaluate(RUNS[n], qrels, 10)["ndcg@10"] for n in RUNS if n in ("BM25", "LSI-48", "RRF(BM25+LSI)")}
fig, ax = plt.subplots(figsize=(8, 3.6)); w = .2; sist = list(verdad_nd)
ax.bar(np.arange(3) - 1.5*w, [verdad_nd[s] for s in sist], w, color=GRIS, label="verdad (humanos)")
for i, (sesgo, c) in enumerate(zip((0.0, 0.4, 0.8), (VERDE, NARANJA, ROJO))):
    sub = jdf[jdf.sesgo == sesgo].set_index("sistema").loc[sist]
    ax.bar(np.arange(3) + (i - 0.5) * w, sub["nDCG@10 (juez)"], w, color=c, label=f"juez sesgo={sesgo} (κ={sub['κ juez vs verdad'].iloc[0]:.2f})")
ax.set_xticks(range(3)); ax.set_xticklabels(sist); ax.set_ylabel("nDCG@10"); ax.set_ylim(0, .95); ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center"); ax.set_title("Un juez sesgado hacia el solapamiento léxico cambia la tabla de resultados")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) Incluso con sesgo 0 los valores **absolutos suben** respecto a la verdad: el juez solo etiqueta el *pool* (top-10 de los tres sistemas), los relevantes fuera del pool desaparecen del IDCG, y eso infla el nDCG (el mismo mecanismo que en 6.2). (2) Al subir el sesgo léxico, los documentos *irrelevantes pero con palabras de la query* se etiquetan como relevantes: las **diferencias entre sistemas se comprimen y se reordenan** (BM25 recupera terreno) y κ cae de 0.8 a ≈ 0.4. Un juez «razonable en promedio» puede sesgar la comparación justo en la dirección que te interesa evaluar (léxico vs semántico).

### 7.2 Plantilla de prompt y celda opcional con API
Buenas prácticas verificables en la literatura de LLM-judge: rúbrica con grados definidos, pedir la etiqueta **después** de un razonamiento corto (o solo etiqueta con `temperature=0` si priorizas coste), barajar el orden cuando compares pares, y medir κ contra humanos.
La celda está **protegida**: solo llama a la API si defines `ANTHROPIC_API_KEY` y `JUDGE_MODEL` (el identificador del modelo lo eliges tú; no lo fijamos porque cambia con el tiempo).

In [ ]:
PROMPT_JUEZ = """Eres un evaluador de relevancia para un buscador de documentación técnica.
Consulta: {query}
Documento: {doc}

Grados: 0 = irrelevante (aunque comparta palabras con la consulta); 1 = tangencial; 2 = responde parcialmente; 3 = responde completamente.
Considera el SIGNIFICADO, no la coincidencia de palabras. Responde SOLO con un dígito 0-3."""

JUDGE_MODEL = os.environ.get("JUDGE_MODEL")
if os.environ.get("ANTHROPIC_API_KEY") and JUDGE_MODEL:
    import anthropic
    client = anthropic.Anthropic()
    def juez_llm(query: str, doc: str) -> int:
        msg = client.messages.create(model=JUDGE_MODEL, max_tokens=5, temperature=0,
                                     messages=[{"role": "user", "content": PROMPT_JUEZ.format(query=query, doc=doc[:1500])}])
        m = re.search(r"[0-3]", msg.content[0].text)
        return int(m.group()) if m else 0
    muestra = [(q, d) for q in list(pool)[:5] for d in pool[q][:4]]
    humano = [int(qrels[q].get(d, 0) > 0) for q, d in muestra]; llm = [int(juez_llm(q_text[q], doc_text[d]) >= 2) for q, d in muestra]
    print(f"κ(LLM, verdad) en {len(muestra)} pares = {cohen_kappa(humano, llm):.2f}")
else:
    print("Sin ANTHROPIC_API_KEY/JUDGE_MODEL: celda omitida (opcional). Prompt de ejemplo:\n" + PROMPT_JUEZ.format(query="cómo configurar ttl", doc="…")[:250])

## 8. Límites teóricos de los embeddings de dimensión fija (LIMIT)

Un buscador *denso* representa cada documento con un vector de dimensión $d$ y cada query con otro; el top-$k$ de una query es el conjunto de los $k$ documentos con mayor producto escalar.
**Pregunta:** ¿cuántos *subconjuntos distintos* de $k$ documentos puede devolver un conjunto de $n$ documentos al variar la query? Si el producto necesita devolver **cualquier** par posible (k = 2) y hay $\binom{n}{2}$ pares, la dimensión no puede ser arbitrariamente pequeña.

**Resultado (Weller, Boratko, Naim, Lee; arXiv ago-2025, ICLR 2026).** Con documentos como vectores unitarios y una query que realiza un subconjunto $S$ con margen $\ge 2\gamma$ sobre el resto, un argumento de empaquetamiento de esferas da:

$$\binom{n}{k}\le\left(1+\tfrac{1}{\gamma}\right)^{d}\ \Longrightarrow\ d\ \ge\ \frac{\log\binom{n}{k}}{\log(1+1/\gamma)}$$

Es una **cota inferior optimista** (ignora que el modelo se entrena con gradiente y que su texto restringe los vectores). Los autores muestran además, optimizando **embeddings libres** (sin pasar por ningún encoder) directamente sobre las qrels de test, que el *tamaño crítico* de corpus $n$ que aún se resuelve crece de forma polinómica con $d$,
y construyen **LIMIT**, un dataset realista y trivialmente simple («¿a quién le gusta X?») donde los embedders de SOTA fallan. Verificaremos la cota y reproduciremos el experimento de embeddings libres en miniatura.

In [ ]:
from math import lgamma, log, ceil
def d_minimo(n: int, k: int, gamma: float = 0.1) -> int:
    """Cota inferior de dimensión: log C(n,k) / log(1+1/γ)  (usa lgamma: C(n,k) enorme)."""
    logC = lgamma(n + 1) - lgamma(k + 1) - lgamma(n - k + 1)
    return ceil(logC / log(1 + 1 / gamma))
filas = [{"n (docs)": f"{n:,}", "k": k, "d ≥ (γ=0.1)": d_minimo(n, k)} for n in (10**4, 10**6, 10**9) for k in (2, 10, 100)]
pd.DataFrame(filas)

> 🔬 **Comprobación contra el paper.** Para $n=10^6$: $k=2$ da $d\ge 12$ y $k=100$ da $d\ge 425$ (con $\gamma=0.1$): son los valores de la Tabla 1 de Weller et al. que reproduce la celda.
> La cota crece solo **logarítmicamente** con $n$ pero **casi linealmente con $k$**: pedir top-100 *exacto* para todas las combinaciones es mucho más exigente que top-2.

### 8.1 Experimento de embeddings libres (miniatura del de los autores)
Optimizamos directamente vectores de documentos y de queries (sin encoder) para que **cada par** de documentos sea el top-2 de alguna query. Buscamos el mayor $n$ que se logra para cada $d$.

In [ ]:
import torch.nn.functional as F
def realizable(n: int, d: int, steps: int = 600, lr: float = 0.05, seed: int = 0, gamma: float = 0.01) -> bool:
    """¿Existen vectores unitarios tales que, para TODO par (i,j), una query tiene a i,j como top-2? (optimiza margen)."""
    g = torch.Generator().manual_seed(seed)
    pairs = torch.combinations(torch.arange(n), 2); m = len(pairs); rows = torch.arange(m)
    D = torch.randn(n, d, generator=g).requires_grad_(); Q = torch.randn(m, d, generator=g).requires_grad_()
    opt = torch.optim.Adam([D, Q], lr=lr)
    mask = torch.ones(m, n, dtype=torch.bool); mask[rows, pairs[:, 0]] = False; mask[rows, pairs[:, 1]] = False
    for _ in range(steps):
        S = F.normalize(Q, dim=1) @ F.normalize(D, dim=1).T
        margin = S.gather(1, pairs).min(1).values - S.masked_fill(~mask, -2).max(1).values
        if bool((margin > 0).all()): return True
        loss = F.relu(gamma - margin).mean(); opt.zero_grad(); loss.backward(); opt.step()
    return False

dims = [2, 3, 4, 5, 6, 8] if FAST_DEV_RUN else [2, 3, 4, 5, 6, 8, 10, 12]
crit, t0 = {}, time.time()
for d in dims:
    best = 0
    for n in range(3, 80):
        if any(realizable(n, d, seed=s) for s in range(2)): best = n
        elif n > best + 3: break
    crit[d] = best
print(f"n crítico por dimensión: {crit}   ({time.time()-t0:.0f}s)")

In [ ]:
dd = np.array(list(crit)); nn = np.array(list(crit.values()))
cubic_paper = lambda d: -10.5322 + 4.0309 * d + 0.0520 * d**2 + 0.0037 * d**3      # ajuste reportado por Weller et al. (v2)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(dd, nn, "o-", color=AZUL, label="este notebook (embeddings libres, 2 semillas)")
ax[0].plot(dd, cubic_paper(dd), "--", color=NARANJA, label="ajuste cúbico del paper (v2)")
ax[0].set_xlabel("dimensión d"); ax[0].set_ylabel("n crítico (todos los pares son top-2)"); ax[0].legend(frameon=False); ax[0].set_title("El corpus que cabe crece polinómicamente con d")
N_TOT = crit[dims[-1]]
pares_tot = N_TOT * (N_TOT - 1) // 2
ax[1].bar(["pares posibles\n(n=%d)" % N_TOT, "pares que un\nvector 2-D puede devolver"], [pares_tot, N_TOT], color=[GRIS, ROJO])
ax[1].set_title("En d=2 solo caben ~n pares adyacentes de C(n,2)"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) El $n$ crítico **crece rápido pero no exponencialmente** con $d$, y nuestras curvas, aun con pocas semillas y pasos, siguen la tendencia del ajuste del paper (no esperes coincidencia exacta: depende de optimizador y pasos).
> (2) En $d=2$ los documentos son puntos de una circunferencia y una query solo puede devolver pares **adyacentes**: de $\binom{n}{2}$ pares posibles, ~$n$ son alcanzables.
> (3) Esto es el **mejor caso** (vectores libres con acceso a las qrels). Un embedder real, que genera los vectores desde texto, lo tiene peor.

> ### 🎨 GEMINI-IMAGEN · ID: `M15-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Por qué dimensión pequeña no puede devolver todos los pares**
>
> **Qué muestra:** Intuición geométrica del límite: 5 documentos como puntos en una circunferencia (d=2); cada query es una dirección y solo devuelve pares adyacentes; 5 de 10 pares son alcanzables.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Pentágono de cinco documentos sobre una circunferencia y una cuadrícula que muestra que solo cinco de los diez pares son el top-2 de alguna query.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas. 16:9.
> Composición con dos paneles lado a lado.
> Panel izquierdo, título «d = 2: 5 documentos en una circunferencia»: una circunferencia gris fina centrada; sobre ella 5 puntos azules etiquetados A, B, C, D, E a ángulos 90°, 18°, −54°, −126°, 162° (pentágono regular). Desde el centro, una flecha naranja larga etiquetada «query u» apunta entre A y B; los puntos A y B quedan resaltados en verde con el texto «top-2 = {A, B}».
> Panel derecho, título «Pares posibles vs. pares alcanzables»: una cuadrícula 5×5 triangular con los 10 pares AB, AC, AD, AE, BC, BD, BE, CD, CE, DE; los 5 pares adyacentes (AB, BC, CD, DE, EA) en verde con una marca ✓ y los otros 5 (AC, AD, BD, BE, CE) en gris tachados con una cruz.
> Debajo, una línea de texto: «Con d = 2 solo 5 de 10 pares son top-2 de alguna query. Más dimensiones → más pares alcanzables».
> Evita: texto ilegible o inventado, más de 5 puntos, sombras 3D, ilustraciones decorativas, flechas ambiguas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M15-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Pentágono de cinco documentos sobre una circunferencia y una cuadrícula que muestra que solo cinco de los diez pares son el top-2 de alguna query.](assets/M15-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
>     A
> E       B        query u apunta entre A y B  -> top-2 = {A,B}
>   D   C          pares alcanzables: AB BC CD DE EA (5 de 10)
> ```

> 👀 **Qué debes observar:** con 5 documentos en un círculo ($d = 2$), una query es una dirección y su top-2 son siempre dos vecinos **adyacentes**: solo 5 de los $\binom{5}{2} = 10$ pares son alcanzables, por mucho que entrenes. Ese es el límite de LIMIT en miniatura: no es un fallo de entrenamiento sino de **geometría**. BM25 no lo sufre porque cada término es su propia dimensión (el vocabulario hace de $d$ enorme), y lo comprobarás en la celda siguiente.

### 8.2 LIMIT en miniatura: BM25 contra un «embedding» de dimensión baja
Réplica de la construcción del paper: $n$ personas, un atributo único por **cada par** de personas («le gusta *kobelu*» solo a las personas $i$ y $j$); la query «¿a quién le gusta *kobelu*?» tiene exactamente 2 relevantes.
Evaluamos BM25 (espacio de dimensión ≈ tamaño del vocabulario) y LSI con dimensión $d$ creciente.

In [ ]:
def genera_limit(n_personas: int = 24, seed: int = SEED):
    rng = random.Random(seed); syl = ["ka", "ko", "mi", "su", "ta", "ne", "lo", "ri", "pa", "ve", "zu", "do"]
    nombres, usados = [], set()
    def palabra():
        while True:
            w = "".join(rng.choice(syl) for _ in range(3))
            if w not in usados: usados.add(w); return w
    gustos = {i: [] for i in range(n_personas)}; qs, qr = [], {}
    for i in range(n_personas):
        for j in range(i + 1, n_personas):
            a = palabra(); gustos[i].append(a); gustos[j].append(a)
            qid = f"lq{i}-{j}"; qs.append({"qid": qid, "lang": "en", "text": f"who likes {a}"}); qr[qid] = {f"p{i}": 1, f"p{j}": 1}
    ds = [{"id": f"p{i}", "lang": "en", "text": f"person {i} likes " + " ".join(rng.sample(gustos[i], len(gustos[i])))} for i in range(n_personas)]
    return ds, qs, qr

L_docs, L_qs, L_qr = genera_limit(24 if FAST_DEV_RUN else 40)
L_ids, L_txt = [d["id"] for d in L_docs], [d["text"] for d in L_docs]
print(f"{len(L_docs)} docs, {len(L_qs)} queries (= todos los C(n,2) pares), {len(L_qr[L_qs[0]['qid']])} relevantes por query")
bm = BM25(tok_simple).fit(L_txt, L_ids); r_bm = build_run(bm.scores, L_qs, L_ids, 10)
res_limit = {"BM25": float(np.mean([recall_at_k(r_bm[q["qid"]], L_qr[q["qid"]], 2) for q in L_qs]))}
curva = {}
for d in (2, 3, 4, 6, 8, 12, 16, len(L_docs) - 1):
    m = LSI(dim=d, seed=SEED).fit(L_txt, L_ids); rr_ = build_run(m.scores, L_qs, L_ids, 10)
    curva[d] = float(np.mean([recall_at_k(rr_[q["qid"]], L_qr[q["qid"]], 2) for q in L_qs]))
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(list(curva), list(curva.values()), "o-", color=AZUL, label="LSI (denso) con dimensión d")
ax.axhline(res_limit["BM25"], color=NARANJA, ls="--", label=f"BM25 (≈ dimensión del vocabulario): {res_limit['BM25']:.2f}")
ax.set_xscale("log", base=2); ax.set_xlabel("dimensión d (escala log)"); ax.set_ylabel("Recall@2"); ax.legend(frameon=False); ax.set_title("LIMIT en miniatura: la dimensión limita al denso, no al léxico")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con $d$ pequeña el Recall@2 del denso es bajo y sube con $d$; BM25 resuelve casi todo porque vive en un espacio de dimensión ≈ tamaño del vocabulario.
> **Cuidado con la conclusión:** este juguete *exagera* (las queries no tienen semántica: son tokens únicos). El paper muestra en el LIMIT real (50 K documentos, 1 000 queries, k = 2) que BM25 llega a 85.7 de Recall@2 (93.6 de Recall@100) mientras que, p. ej., Gemini Embed (3072-d) llega a 1.6 de Recall@2 y 10.0 de Recall@100 (Tabla 5 del paper, v2, a oct-2026), y que un multi-vector (GTE-ModernColBERT) queda en medio (23.1 de Recall@2).
> Lectura madura: **la dimensión importa cuando la consulta pide combinaciones arbitrarias** (instrucciones, razonamiento, filtros implícitos). Para búsqueda temática típica, los embeddings siguen funcionando; por eso el híbrido (módulo 13) y el reranking son la respuesta práctica.

## 9. Contaminación y sobreajuste al leaderboard

**Contaminación** = el modelo vio datos de evaluación durante el entrenamiento (directa: las queries/documentos de test; indirecta: paráfrasis o el mismo corpus de Wikipedia en varios benchmarks).
**Sobreajuste al leaderboard** = aunque no haya fuga, elegir el modelo/hiperparámetros por la puntuación en un test público fijo lo convierte en *validation set*.

Lo que se sabe (y lo que no):
- Los autores de **E5-Mistral** reportan haber buscado coincidencias exactas entre su entrenamiento y los test de MTEB: encontraron muy pocas (p. ej. 4 preguntas de DBpedia también en TriviaQA) y juzgaron su efecto insignificante; también señalan que varios datasets de retrieval comparten el mismo corpus de Wikipedia. Es una auditoría **de un modelo**, no del leaderboard.
- Es habitual entrenar con los *train splits* de los datasets de MTEB. No es trampa, pero hace que «zero-shot» deje de serlo; la leaderboard de MTEB muestra un porcentaje «zero-shot» por modelo (fracción de tareas cuyo train split no se usó), que conviene mirar.
- **RTEB** (beta, oct-2025) responde con datasets privados: su paper reporta 48 datasets (28 privados) y una correlación débil de rankings con BEIR/MTEB (ρ = 0.38). 🟡 Es evidencia de que los rankings públicos y privados difieren, no una prueba de cuánto hay de contaminación.

### 9.1 Experimento: qué aspecto tiene una fuga y cómo detectarla
Simulamos la fuga más simple: «expandir» los documentos con las **queries públicas** (como si el entrenamiento las hubiera visto). Medimos el salto en las queries públicas vs las privadas, y un detector de solapamiento de n-gramas.

In [ ]:
rng = random.Random(SEED); qs_all = [q["qid"] for q in queries]; rng.shuffle(qs_all)
publicas, privadas = set(qs_all[: len(qs_all) // 2]), set(qs_all[len(qs_all) // 2:])
expand = dict(doc_text)
for q in publicas:
    for d in qrels[q]: expand[d] = expand[d] + " " + q_text[q]          # fuga: el "entrenamiento" vio la query
def nd_sub(model, subset): 
    run = build_run(model.scores, [q for q in queries if q["qid"] in subset], doc_ids, 10)
    return float(np.mean([ndcg_at_k(run[q], qrels[q], 10) for q in subset]))
limpio = BM25(tok_simple).fit(doc_texts, doc_ids); sucio = BM25(tok_simple).fit([expand[d] for d in doc_ids], doc_ids)
def ngramas(texto, n): t = tok_simple(texto); return {tuple(t[i:i + n]) for i in range(len(t) - n + 1)}
def frac_contaminada(train_texts, test_texts, n=3):
    tr = set().union(*[ngramas(t, n) for t in train_texts]); return np.mean([bool(ngramas(t, n) & tr) for t in test_texts])
res_c = {"limpio": (nd_sub(limpio, publicas), nd_sub(limpio, privadas)), "contaminado": (nd_sub(sucio, publicas), nd_sub(sucio, privadas))}
det = {"ref. limpia": frac_contaminada(doc_texts, [q_text[q] for q in publicas]), "tras la fuga": frac_contaminada([expand[d] for d in doc_ids], [q_text[q] for q in publicas])}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5)); x = np.arange(2); w = .35
ax[0].bar(x - w/2, res_c["limpio"], w, color=AZUL, label="modelo limpio"); ax[0].bar(x + w/2, res_c["contaminado"], w, color=ROJO, label="modelo contaminado")
ax[0].set_xticks(x); ax[0].set_xticklabels(["queries PÚBLICAS\n(vistas en la fuga)", "queries PRIVADAS"]); ax[0].set_ylabel("nDCG@10"); ax[0].legend(frameon=False); ax[0].set_title("La fuga infla solo lo público")
ax[1].bar(list(det), list(det.values()), color=[VERDE, ROJO]); ax[1].set_ylabel("fracción de queries con un 3-grama en el train"); ax[1].set_title("Detector de solapamiento de n-gramas")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** El modelo contaminado «mejora» muchísimo en lo público y **nada** en lo privado (aquí incluso empeora: el texto añadido es ruido para las demás queries): la brecha público–privado es la huella de la fuga. El detector de n-gramas la señala (con n=3 por la brevedad de nuestras queries; en documentos largos se usan n≈13).
> Qué hacer: (1) mantén un **set privado** propio y no lo uses para decidir; (2) reporta la brecha público–privado; (3) con modelos de terceros, desconfía de saltos grandes en un benchmark público sin mejora en el privado; (4) fecha las cifras.

## 10. Drift: tu evaluación también caduca
Hay tres drifts distintos: **de queries** (los usuarios empiezan a preguntar otra cosa), **de corpus** (llegan documentos nuevos con vocabulario nuevo) y **de modelo** (cambias de versión del embedder o del tokenizer).
El golden set congelado es un *termómetro de laboratorio*: no ve que el tráfico real cambió. Soluciones: ponderar por tráfico, un **set canario** pequeño ejecutado en cada despliegue, y monitorizar la distribución de entradas (p. ej. PSI).

In [ ]:
def psi(p, q, eps=1e-6):
    """Population Stability Index entre dos distribuciones discretas. Regla práctica: <0.1 estable, 0.1–0.25 vigilar, >0.25 drift."""
    p, q = np.clip(np.asarray(p, float), eps, None), np.clip(np.asarray(q, float), eps, None); p, q = p / p.sum(), q / q.sum()
    return float(np.sum((p - q) * np.log(p / q)))

langs = ["es", "en", "code"]; nd_q = dict(zip(per["BM25"]["qids"], per["BM25"]["ndcg"]))
por_lang = {l: np.array([nd_q[q] for q in nd_q if q_lang[q] == l]) for l in langs}
rng = np.random.default_rng(SEED); semanas = np.arange(12); base_mix = np.array([.45, .45, .10])
golden_fijo = evaluate(RUNS["BM25"], qrels, 10)["ndcg@10"]
filas = []
for s in semanas:
    code_share = .10 + .30 * s / 11                       # el tráfico migra hacia consultas de código
    mix = np.array([(1 - code_share) / 2, (1 - code_share) / 2, code_share])
    muestra = [rng.choice(por_lang[l], int(round(300 * m))) for l, m in zip(langs, mix)]
    filas.append({"semana": s, "nDCG tráfico real": np.concatenate(muestra).mean(), "nDCG golden fijo": golden_fijo, "PSI mezcla": psi(base_mix, mix)})
dr = pd.DataFrame(filas)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(dr.semana, dr["nDCG golden fijo"], "--", color=GRIS, label="golden set fijo (no se entera)"); ax[0].plot(dr.semana, dr["nDCG tráfico real"], "o-", color=ROJO, label="ponderado por tráfico real")
ax[0].set_xlabel("semana"); ax[0].set_ylabel("nDCG@10 de BM25"); ax[0].legend(frameon=False); ax[0].set_title("La calidad percibida cae aunque el 'benchmark' no cambie")
ax[1].plot(dr.semana, dr["PSI mezcla"], "o-", color=AZUL); ax[1].axhline(.1, color=NARANJA, ls="--", label="vigilar (0.1)"); ax[1].axhline(.25, color=ROJO, ls="--", label="drift (0.25)")
ax[1].set_xlabel("semana"); ax[1].set_ylabel("PSI de la mezcla de idiomas"); ax[1].legend(frameon=False); ax[1].set_title("El PSI da la alarma antes de que se queje nadie")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Cuando el tráfico migra hacia las consultas donde el sistema es peor (aquí, código con tokens `snake_case`), la métrica **ponderada por tráfico** cae mientras el golden fijo permanece plano.
> El PSI cruza los umbrales de vigilancia/drift semanas antes de que la caída sea evidente en las quejas. Lo desarrollamos en producción en el módulo 18 (monitoreo, A/B).

## 11. Análisis de errores: de «0.52» a «qué arreglar»
Una media esconde *dónde* falla el sistema. Receta: (1) **slices** (idioma, longitud, solapamiento léxico con el relevante, tipo de intención); (2) ordenar queries por pérdida; (3) leer 20 fallos y **etiquetar la causa**; (4) cuantificar cada causa y priorizar.

In [ ]:
filas = [{"qid": q, "lang": q_lang[q], **{n: per[n]["ndcg"][i] for n in RUNS}} for i, q in enumerate(per["BM25"]["qids"])]
ea = pd.DataFrame(filas)
tab = ea.groupby("lang")[list(RUNS)].mean().round(3)
fig, ax = plt.subplots(figsize=(6.5, 2.8)); im = ax.imshow(tab.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(tab.shape[1])); ax.set_xticklabels(tab.columns, fontsize=8); ax.set_yticks(range(tab.shape[0])); ax.set_yticklabels(tab.index)
for i in range(tab.shape[0]):
    for j in range(tab.shape[1]): ax.text(j, i, f"{tab.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("nDCG@10 por idioma de la query y sistema"); ax.grid(False); plt.colorbar(im, ax=ax, shrink=.8); plt.tight_layout(); plt.show()
print("Peores queries para BM25:"); print(ea.sort_values("BM25").head(5)[["qid", "lang", "BM25", "LSI-48"]].to_string(index=False))

> 👀 **Qué debes observar.** El promedio global oculta que las consultas de **código** son el punto débil de BM25 (fila `code`), y que el denso las rescata parcialmente. Una media de 0.5 «mezcla» un 0.8 con un 0.1.
> Antes de comprar un modelo más grande, mira *por qué* falla la fila roja.

In [ ]:
# Diagnóstico de causa: ¿las queries de código fallan porque snake_case NO se parte? (tokenización con criterio de dominio)
bm25_codigo = BM25(tok_code).fit(doc_texts, doc_ids)
run_tc = build_run(bm25_codigo.scores, queries, doc_ids, 100); per_tc = evaluate_per_query(run_tc, qrels, 10)
mask = np.array([q_lang[q] == "code" for q in per_tc["qids"]])
print(f"BM25 tokenización simple  | code: {per['BM25']['ndcg'][mask].mean():.3f}  todo: {per['BM25']['ndcg'].mean():.3f}")
print(f"BM25 parte snake_case     | code: {per_tc['ndcg'][mask].mean():.3f}  todo: {per_tc['ndcg'].mean():.3f}")
d_tc = paired_bootstrap(per_tc["ndcg"][mask], per["BM25"]["ndcg"][mask])
print(f"Δ en code = {d_tc['diff']:+.3f}  IC95 [{d_tc['ci_low']:+.3f}, {d_tc['ci_high']:+.3f}]  p={d_tc['p_value']:.4f}")

> 🔁 **Conexión con tokenización (módulos 01–05).** El fallo de BM25 en código **no es de «semántica»**: es de *tokenización* (`set_ttl` es un único token léxico que nunca coincide con «ttl»).
> Un buen análisis de errores te devuelve al módulo correcto. Si la causa fuera morfología del español, la acción sería lematizar o usar subwords; si fuera vocabulario, un embedder multilingüe.

In [ ]:
# Slice por "dificultad léxica": fracción de términos de la query presentes en algún doc relevante
def solape(q):
    qt = set(tok_simple(q_text[q])); rel_tokens = set().union(*[set(tok_simple(doc_text[d])) for d in qrels[q]])
    return len(qt & rel_tokens) / max(1, len(qt))
ea["solape"] = [solape(q) for q in ea.qid]; ea["bin"] = pd.cut(ea.solape, [-.01, .34, .67, 1.0], labels=["bajo", "medio", "alto"])
g = ea.groupby("bin", observed=True)[["BM25", "LSI-48", "RRF(BM25+LSI)"]].mean()
ax = g.plot(kind="bar", color=[AZUL, NARANJA, VERDE], figsize=(7, 3.4), rot=0); ax.set_xlabel("solape léxico query–relevantes"); ax.set_ylabel("nDCG@10")
ax.set_title("El denso gana donde el léxico no ve (solape bajo)"); ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()
print(ea.groupby("bin", observed=True).size().rename("nº queries").to_dict())

> 👀 **Qué debes observar.** Cuanto menor es el solapamiento léxico, más cae BM25 y más aporta un componente semántico: la justificación empírica del retrieval híbrido (módulo 13) sale del **análisis por slices**, no de la media.
> Ojo con el tamaño de cada slice: con pocas queries por bin, **calcula IC también por slice** antes de actuar.

## 🗺️ Mapa de madurez (estado a oct-2026)
Ya has implementado las métricas, la estadística, el golden set, el juez LLM, LIMIT, la contaminación, el drift y el análisis de errores; la tabla separa lo consolidado de lo que sigue en debate (🟡) antes de ver cómo lo hacen los equipos en producción.


| Técnica / práctica | Madurez | Quién / dónde se usa | Fuente |
|---|---|---|---|
| nDCG@10, Recall@k, MRR@10 con qrels fijos | 🟢 sólido | BEIR, MTEB (retrieval usa nDCG@10 como métrica principal), TREC | Thakur et al. 2021; Muennighoff et al. 2022 |
| Benchmarks públicos multi-dominio (BEIR, MTEB/MMTEB, MIRACL) | 🟢 sólido (con límites de contaminación) | Estándar de facto para comparar embedders | Thakur 2021; Enevoldsen 2025; Zhang 2022 |
| Bootstrap / tests pareados sobre queries | 🟢 sólido en IR clásico, 🟡 poco adoptado en papers de embeddings | Comunidad TREC/SIGIR | Smucker et al. 2007; Urbano et al. 2019 |
| Golden sets propios + anotación con acuerdo κ | 🟢 sólido | Equipos de búsqueda en producción | práctica de la industria |
| LLM-as-judge para relevancia | 🟡 emergente y **debatido** | Investigación (Bing/TREC) y equipos de evaluación | Thomas et al. 2023; Soboroff 2024; Clarke & Dietz 2024 |
| Benchmarks con parte privada contra overfitting (RTEB) | 🟡 emergente (beta lanzada oct-2025) | Hugging Face / comunidad MTEB | RTEB (HF blog, 2025) |
| Límite teórico de la dimensión del embedding (LIMIT) | 🟡 emergente (resultado teórico sólido; relevancia práctica en discusión) | Investigación (Google DeepMind / JHU) | Weller et al., ICLR 2026 |
| Monitorización de drift con set canario | 🟢 sólido como práctica; 🟡 como tooling específico de embeddings | Producción | módulo 18 |

## 🏭 En producción
- **Evaluación continua, no puntual.** Equipos de búsqueda en producción mantienen (i) un golden set versionado, (ii) un set **canario** pequeño que corre en cada cambio de modelo/índice/tokenizer, y (iii) métricas online (clics, reformulaciones, abandono) que se contrastan periódicamente con las offline. Si offline y online se desacoplan, tu golden set está obsoleto.
- **MTEB/MMTEB como filtro, tu golden set como decisión.** Los leaderboards sirven para preseleccionar 3–5 modelos; la decisión se toma en datos propios. Los proveedores de embeddings (OpenAI, Google, Cohere, Voyage, Jina…) publican resultados en benchmarks públicos; trátalos como *marketing técnico* hasta reproducirlos en tu dominio.
- **Anotación asistida por LLM** (triaje, sugerencias) con humanos en el bucle para el set de decisión, medido con κ y re-calibrado al cambiar de modelo-juez.
- **Costes de re-embedding.** Evaluar un modelo nuevo implica re-indexar el corpus entero; presupuesta ese coste *antes* (módulo 18) y evalúa primero en una muestra con el mismo muestreo de documentos relevantes + negativos duros.
- **Informe estándar:** métrica principal con IC, diferencia pareada vs baseline con IC y p corregido, tabla por slices, brecha público–privado y la versión exacta de dataset, modelo, tokenizer y semilla.

## 🧠 Secretos de la élite
1. **Siempre diferencia pareada con IC, nunca dos IC separados.** Las queries comparten dificultad; el test pareado detecta mejoras que los IC por sistema esconden (experimento 5.3).
2. **Un golden set de 100 queries solo distingue diferencias grandes.** La semi-anchura del IC cae como $1/\sqrt n$; calcula el $n$ necesario con *tu* $\sigma$ (sección 5.3) antes de prometer «+1 punto».
3. **Los empates y duplicados son una fuente silenciosa de varianza.** Fija el desempate (orden del corpus) y deduplica el ranking; es el primer sitio donde busco diferencias entre dos implementaciones de la «misma» métrica.
4. **Qrels incompletos castigan a lo nuevo.** BEIR reconoce sesgo léxico en varios datasets (Thakur et al., sección 6); al evaluar un sistema distinto de los que generaron los qrels, añade su top-k al pool o corrige con juicios nuevos (en TREC-COVID, anotar juicios faltantes subió a los modelos densos).
5. **Mide la brecha público–privado.** Si tu modelo sube 3 puntos en el benchmark público y 0 en tu set privado, has aprendido el benchmark. Por eso existe RTEB.
6. **LLM-as-judge sesgado hacia el léxico favorece a BM25.** Alaofi et al. muestran que meter palabras de la query en texto irrelevante engaña a muchos LLMs; valida el juez con casos adversariales antes de confiar en él.
7. **La dimensión es un presupuesto, no un parámetro libre.** Si tu producto pide combinaciones arbitrarias (instrucciones, filtros implícitos), LIMIT te dice que un vector único tiene un techo; multi-vector, híbrido y reranking existen por eso.
8. **El análisis de errores por slices decide la hoja de ruta.** En nuestro corpus la «mejora de embedder» que parecía necesaria era un arreglo de tokenización de código (sección 11).

## ⚠️ Errores comunes
- **Evaluar sobre queries que usaste para elegir hiperparámetros** (no reservar test ciego).
- **Promediar sin mirar la varianza**: 0.52 vs 0.50 con σ=0.35 y 70 queries es ruido.
- **Comparar sistemas con k, métrica o corpus distintos** (p. ej. comparar tu nDCG@10 con el de un paper que usó otro split o qrels).
- **Dividir por el número de relevantes devueltos en vez de por el total** (Recall inflado) o **no normalizar embeddings** al comparar coseno vs producto escalar.
- **Descartar silenciosamente las queries sin resultados** (el sistema que no responde sale ganando).
- **Usar el mismo LLM como generador de queries, juez y sistema** (circularidad).
- **Citar cifras de leaderboards sin fecha ni versión** del benchmark.
- **Confiar en nDCG@10 si el producto muestra 3 resultados** (usa la métrica que corresponde a la UI, p. ej. nDCG@3 / MRR).

## 📝 Autoevaluación
Intenta responder antes de abrir cada respuesta.

<details><summary><b>1. (Cálculo)</b> Ranking <code>[a, b, c, d]</code>, relevancias <code>{b: 2, d: 1, e: 3}</code>. Calcula nDCG@4 (ganancia lineal).</summary>

DCG = $2/\log_2 3 + 1/\log_2 5 = 1.262 + 0.431 = 1.693$. IDCG (mejor orden: $e{=}3, b{=}2, d{=}1$) $= 3/1 + 2/\log_2 3 + 1/\log_2 4 = 3 + 1.262 + 0.5 = 4.762$. nDCG@4 $= 0.356$. Observa que *e* no aparece en el ranking pero sí cuenta en el IDCG: el recall perdido penaliza.
</details>

<details><summary><b>2. (Cálculo)</b> Tu σ entre queries es 0.30. ¿Cuántas queries necesitas para un IC95 % de semi-anchura ±0.02 sobre una media?</summary>

$n\approx(1.96\cdot\sigma/h)^2=(1.96\cdot0.30/0.02)^2\approx 864$ queries. Si comparas dos sistemas de forma *pareada*, lo relevante es la σ de la diferencia (normalmente menor), así que necesitas menos.
</details>

<details><summary><b>3. (Diagnóstico)</b> El embedder nuevo sube +0.02 nDCG@10 sobre 60 queries y el IC95 % pareado es [−0.03, +0.07]. ¿Despliegas?</summary>

No hay evidencia de mejora: el IC incluye 0. Opciones: ampliar el golden set (necesitas ~4× queries para duplicar la precisión), mirar slices donde se espera la ganancia, o decidir con otros criterios (coste, latencia). Declarar «mejora» sería *p-hacking* por falta de potencia.
</details>

<details><summary><b>4. (Diagnóstico)</b> Tu nuevo modelo denso supera a BM25 en tu leaderboard pero los usuarios no lo notan. Da tres hipótesis y cómo las comprobarías.</summary>

(a) Qrels sesgados o incompletos → añade el pool del nuevo sistema y re-anota una muestra. (b) El golden set no representa el tráfico (drift de queries) → reponderar por tráfico / PSI. (c) Contaminación o sobreajuste al set → brecha público–privado. (d) La UI solo muestra 3 resultados y optimizaste nDCG@10 → mide nDCG@3/MRR.
</details>

<details><summary><b>5. (Transferencia)</b> Tienes que evaluar un buscador de contratos en español con 0 anotaciones. Diseña el plan en 5 pasos.</summary>

(1) Muestra de queries de logs/expertos estratificada por tipo de cláusula; (2) pooling de top-20 de BM25 + 2 embedders; (3) anotación humana con guía de 4 grados sobre 300 queries y doble anotación del 15 % (κ ≥ 0.7); (4) LLM-judge solo para triaje, validado contra humanos; (5) congelar v1.0 con test ciego, métrica nDCG@10 con IC bootstrap, test pareado entre candidatos con Holm, canario semanal para drift.
</details>

<details><summary><b>6. (Transferencia)</b> ¿Por qué LIMIT no implica que «los embeddings no sirven»? ¿Qué cambia en tu arquitectura?</summary>

La cota habla de realizar *todas* las combinaciones posibles de top-$k$ y de un margen dado; las consultas temáticas reales ocupan una fracción minúscula de ese espacio y los embeddings funcionan bien ahí. Pero si el producto admite instrucciones o combinaciones arbitrarias (lógica, filtros), el vector único tiene un techo: usa híbrido (BM25 + denso), multi-vector o reranker, y filtra por metadatos antes del ANN.
</details>

<details><summary><b>7. (Diagnóstico)</b> El Recall@10 de BM25 en tu set es 0.81 y el de tu HNSW sobre el mismo embedder es 0.78. ¿Qué evalúa cada número?</summary>

Ambos son recall de **relevancia** sobre tus qrels. El «recall del ANN» frente a fuerza bruta (módulo 14) es otra cosa: mide cuánto del top-10 exacto recupera el índice aproximado. Pérdida total = error del embedder (qrels) más error del índice (ANN).
</details>

## 📚 Referencias
**Evaluación y benchmarks**
- Thakur, Reimers, Rücklé, Srivastava, Gurevych (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models*. NeurIPS 2021 Datasets & Benchmarks. <https://arxiv.org/abs/2104.08663>
- Zhang et al. (2022). *Making a MIRACL: Multilingual Information Retrieval Across a Continuum of Languages*. TACL 11 (2023). <https://arxiv.org/abs/2210.09984>
- Muennighoff, Tazi, Magne, Reimers (2022). *MTEB: Massive Text Embedding Benchmark*. EACL 2023. <https://arxiv.org/abs/2210.07316>
- Enevoldsen et al. (2025). *MMTEB: Massive Multilingual Text Embedding Benchmark*. ICLR 2025. <https://arxiv.org/abs/2502.13595>
- RTEB (2025). *Introducing RTEB: A New Standard for Retrieval Evaluation* (Hugging Face blog) <https://huggingface.co/blog/rteb> y *RTEB: An Overfitting-Resistant Benchmark for Embedding Model Evaluation* (NeurIPS 2026 poster) <https://neurips.cc/virtual/2026/poster/139103>
- Wadden et al. (2020). *Fact or Fiction: Verifying Scientific Claims* (SciFact). EMNLP 2020. <https://arxiv.org/abs/2004.14974>
- Wang, Wang, Li (2013). *A Theoretical Analysis of NDCG Type Ranking Measures*. COLT 2013. <https://arxiv.org/abs/1304.6480>
- Bassani (2022). *ranx: A Blazing-Fast Python Library for Ranking Evaluation and Comparison*. ECIR 2022. <https://github.com/AmenRa/ranx>
- Craswell, Mitra, Yilmaz, Campos, Lin (2021). *MS MARCO: Benchmarking Ranking Models in the Large-Data Regime*. SIGIR 2021. <https://arxiv.org/abs/2105.04021>
- Arabzadeh, Vtyurina, Yan, Clarke (2021). *Shallow pooling for sparse labels*. Information Retrieval Journal 25 (2022). <https://arxiv.org/abs/2109.00062>
- Krichene & Rendle (2020). *On Sampled Metrics for Item Recommendation*. KDD 2020. <https://research.google/pubs/on-sampled-metrics-for-item-recommendation/> (ver `recsys-course/02_evaluation`)

**Estadística en IR**
- Smucker, Allan, Carterette (2007). *A Comparison of Statistical Significance Tests for Information Retrieval Evaluation*. CIKM 2007. DOI 10.1145/1321440.1321528
- Sakai (2006). *Evaluating Evaluation Metrics based on the Bootstrap*. SIGIR 2006. DOI 10.1145/1148170.1148261
- Urbano, Lima, Hanjalic (2019). *Statistical Significance Testing in Information Retrieval: An Empirical Analysis of Type I, Type II and Type III Errors*. SIGIR 2019. <https://arxiv.org/abs/1905.11096>

**LLM-as-judge**
- Thomas, Spielman, Craswell, Mitra (2023). *Large language models can accurately predict searcher preferences*. SIGIR 2024. <https://arxiv.org/abs/2309.10621>
- Faggioli et al. (2023). *Perspectives on Large Language Models for Relevance Judgment*. ICTIR 2023. <https://arxiv.org/abs/2304.09161>
- Soboroff (2024). *Don't Use LLMs to Make Relevance Judgments*. Information Retrieval Research 1(1) (2025). <https://arxiv.org/abs/2409.15133>
- Clarke & Dietz (2024). *LLM-based relevance assessment still can't replace human relevance assessment*. Preprint. <https://arxiv.org/abs/2412.17156>
- Alaofi, Thomas, Scholer, Sanderson (2024). *LLMs can be Fooled into Labelling a Document as Relevant* (título completo: «…(best café near me; this paper is perfectly relevant)»). SIGIR-AP 2024. <https://arxiv.org/abs/2501.17969>
- Zheng et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. NeurIPS 2023 D&B. <https://arxiv.org/abs/2306.05685>

**Límites de los embeddings y contaminación**
- Weller, Boratko, Naim, Lee (2025; v2 mar-2026, ICLR 2026). *On the Theoretical Limitations of Embedding-Based Retrieval*. <https://arxiv.org/abs/2508.21038>
- Reimers & Gurevych (2021). *The Curse of Dense Low-Dimensional Information Retrieval for Large Index Sizes*. ACL 2021. <https://arxiv.org/abs/2012.14210>
- Wang et al. (2024). *Improving Text Embeddings with Large Language Models* (E5-Mistral; sección *Test Set Contamination Analysis* sobre MTEB). ACL 2024. <https://arxiv.org/abs/2401.00368>

**Material del curso**: [`recsys-course/02_evaluation`](../../recsys-course/02_evaluation/README.md) · módulos 00, 10, 12, 13, 14 y 18.